# CoRe-Prompt — version (Colab free tier, v1.2.0)

**Run:** *Runtime ▸ Change runtime type ▸ T4 GPU* → *Runtime ▸ Run all*. Nothing needs editing. Allow Google Drive access.
Results go to `MyDrive/coreprompt_quick` (separate from the full run); the data cache of the full run in `MyDrive/coreprompt_outputs/data_cache` is reused, so nothing is downloaded again if it exists.

All five requirements run with a reduced budget (same code, same methods, same figures and tables as the full version):

| # | Requirement | 1-hour budget | Full version |
|---|---|---|---|
| 1 | PromptSmooth reproduction | 100 test images, σ ∈ {0.25, 0.5, 1.0}, n = 500 noise samples (max radius 2.2σ), 20 epochs | 500 images, 4 σ, n = 2,000, 50 epochs |
| 2 | CoRe-Prompt (± TTA) | same images and noise as the baselines; 2,000 test images for noisy-vote accuracy | 7,180 images |
| 3 | Adaptive attacks | 50 images, 10-step PGD, EOT 4 | 100 images, 20 steps, EOT 8 |
| 4 | Ablation + sensitivity | σ = 0.5, 100 images, 9 prompts | 200 images, 24 prompts |
| 5 | Generalisation (CLIP; BloodMNIST) | 100 images, σ ∈ {0.25, 0.5}, n = 500; 1,000 images for accuracy | 200 images, 3 σ, n = 1,000 |

Estimated time on a T4: 45–60 min (first run without a data cache: +5 min). Every stage is checkpointed; after a disconnect, *Run all* resumes. Smaller samples give wider confidence intervals: the report shows them.

In [ ]:
# GPU check: stops here on a CPU runtime
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all. '
                       'If T4 is unavailable, the free GPU quota is used up: try again later.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Storage on Google Drive (falls back to local disk if Drive is not mounted)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/coreprompt_quick'
    SHARED_DATA = '/content/drive/MyDrive/coreprompt_outputs/data_cache'   # reuse the full run's data cache
except Exception as e:
    print('Drive not mounted (', e, ') -> results kept locally and lost on disconnect')
    OUT_DIR = '/content/outputs'
    SHARED_DATA = None
DATA_DIR = SHARED_DATA or os.path.join(OUT_DIR, 'data_cache')
SMOKE_DIR = os.path.join(OUT_DIR, 'smoke')
for d in (OUT_DIR, DATA_DIR, SMOKE_DIR):
    os.makedirs(d, exist_ok=True)
os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs('/content/CoRe-Prompt/coreprompt', exist_ok=True); os.makedirs('/content/CoRe-Prompt/tests', exist_ok=True)
os.chdir('/content/CoRe-Prompt')
print('outputs ->', OUT_DIR)

## Install dependencies
(installed into the same Python interpreter that runs the stages; torch/torchvision are preinstalled on Colab)

In [ ]:
import sys
!{sys.executable} -m pip -q install "transformers==4.46.3" medmnist scipy pandas matplotlib psutil pytest requests jinja2
!{sys.executable} -c "import torch, transformers, medmnist; print('torch', torch.__version__, '| transformers', transformers.__version__, '| medmnist', medmnist.__version__)"

## Write the `coreprompt` package
The same files as in the GitHub repository, embedded so the notebook needs no clone.

In [ ]:
%%writefile coreprompt/__init__.py
"""CoRe-Prompt: consistency- and radius-regularised prompt learning for certified robustness of medical VLMs."""
import os

# transformers must not import TensorFlow/JAX (slow start-up, extra RAM, misleading CUDA warnings on Colab)
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_JAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

__version__ = "1.2.0"

In [ ]:
%%writefile coreprompt/config.py
"""Experiment configuration.

One budget ("colab") is used for every experiment in the paper. It is sized so that the whole pipeline -
baseline reproduction, proposed method, adaptive attacks, ablation/sensitivity, generalisation, full-test-set
evaluation and efficiency - completes in one Google Colab free-tier session (T4 GPU, ~12 GB RAM), with every stage
checkpointed and resumable. Each experiment is a light-weight version of its full-scale counterpart (Table "budget"
in the paper): fewer Monte-Carlo samples than PromptSmooth (n=2,000 vs 10,000), and nested subsets of the test
split for secondary experiments. The "smoke" preset runs every stage on a handful of images in a few minutes and
is executed automatically before the real run to catch errors early.
"""
from dataclasses import dataclass, field, asdict
from typing import List


@dataclass
class Config:
    # ---------------- data / model ----------------
    dataset: str = "pathmnist"          # pathmnist (= Kather colon, NCT-CRC-HE) | bloodmnist
    img_size: int = 224                 # MedMNIST+ size; 224 keeps native resolution (falls back to 128 if RAM is short)
    model: str = "plip"                 # plip | clip
    shots: int = 16                     # few-shot images per class (PromptSmooth: 16-shot)
    n_test: int = 500                   # test images certified (PromptSmooth: 500)
    n_full: int = 0                     # images of the complete test split to load (None = all; 0 = not needed)
    full_votes: int = 16                # noisy votes per image in the full-test-set evaluation
    seed: int = 0

    # ---------------- prompt learning ----------------
    n_ctx_init: str = "an H&E noisy image patch of"   # noisy-context initialisation (PromptSmooth, Table 4)
    epochs: int = 50                    # PromptSmooth few-shot: 50 epochs
    lr: float = 0.002                   # PromptSmooth few-shot: SGD, lr 0.002
    batch: int = 16                     # PromptSmooth few-shot: batch 16
    m_train: int = 4                    # noise draws per image per step (Jeong & Shin m=2; MACER k=16)

    # ---------------- CoRe-Prompt objective ----------------
    lam_con: float = 10.0               # KL weight lambda; Jeong & Shin (2020) use lambda in {2,5,10,20}
    eta_ent: float = 0.5                # entropy weight eta; Jeong & Shin weight KL and entropy equally (eta=lambda),
                                        # we decouple them (eta=lambda is tested in the sensitivity study)
    mu_rad: float = 1.0                 # radius-hinge weight (MACER lambda; MACER uses 3-16 for full networks)
    gamma_rad: float = 4.0              # hinge threshold; MACER: gamma = 2(eps+eps~)/sigma, i.e. target radius
                                        # gamma*sigma/2 = 2 sigma (n=2,000 certifies at most 2.70 sigma); MACER default 8
    beta_rad: float = 1.0               # inverse temperature on logits already scaled by CLIP's logit scale s (~100);
                                        # MACER uses beta=16 on ResNet logits

    # ---------------- randomized smoothing ----------------
    sigmas: List[float] = field(default_factory=lambda: [0.1, 0.25, 0.5, 1.0])   # PromptSmooth sigmas
    n0: int = 64                        # selection samples
    n: int = 2000                       # estimation samples (PromptSmooth: 10,000; reduced for the free tier)
    alpha: float = 0.001                # failure probability (PromptSmooth / Cohen)
    radii: List[float] = field(default_factory=lambda: [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])
    cert_batch: int = 250               # noisy copies per forward pass

    # ---------------- zero-shot PromptSmooth (test-time) ----------------
    tta_T: int = 100                    # PromptSmooth: T=100 noisy copies
    tta_steps: int = 1                  # PromptSmooth: one gradient step
    tta_lr: float = 5e-3                # not reported by PromptSmooth; TPT (Shu et al. 2022) setting

    # ---------------- attacks ----------------
    n_attack: int = 100
    linf_eps: List[float] = field(default_factory=lambda: [1 / 255, 2 / 255, 4 / 255])
    l2_eps: List[float] = field(default_factory=lambda: [0.25, 0.5, 1.0])
    pgd_steps: int = 20
    eot: int = 8
    predict_n: int = 100                # Cohen PREDICT samples for empirical evaluation of smoothed classifiers

    # ---------------- io ----------------
    out_dir: str = "outputs"            # set to a Google Drive folder in Colab for persistence
    data_dir: str = None                # shared data cache (default OUT_DIR/data_cache)
    data_url: str = None                # override of the MedMNIST+ .npz URL (used by the offline tests)
    grid: str = "full"                  # sensitivity grid: full | medium (1-hour run) | small (smoke test)

    def to_dict(self):
        return asdict(self)


PRESETS = {
    "colab": dict(),                                                    # the paper's budget
    "smoke": dict(n_test=8, n_full=64, full_votes=2, n0=16, n=100, epochs=2, n_attack=4, pgd_steps=2, eot=2,
                  predict_n=20, sigmas=[0.25, 0.5], tta_T=8, cert_batch=50, grid="small"),
    # ~1 hour on a free T4: 3 sigmas, 100 certified images, n = 500 Monte-Carlo samples, 20 training epochs
    "quick": dict(sigmas=[0.25, 0.5, 1.0], n_test=100, n0=32, n=500, epochs=20, tta_T=64, full_votes=8,
                  n_attack=50, pgd_steps=10, eot=4, predict_n=50, grid="medium"),
}

# Stage-specific budgets of a preset (applied after the preset's general values).
PRESET_STAGES = {
    "quick": {
        "fulltest": dict(n_full=2000),
        "ablation": dict(n_test=100),
        "attacks": dict(n_attack=50),
        "gen_clip": dict(sigmas=[0.25, 0.5], n_test=100, n=500, n_full=1000),
        "gen_blood": dict(sigmas=[0.25, 0.5], n_test=100, n=500, n_full=1000),
    },
}

# Per-stage budgets (light-weight versions of each experiment; main certification uses the full budget).
GEN_BUDGET = dict(n_test=200, n=1000, sigmas=[0.25, 0.5, 1.0])
STAGES = {
    "sanity": {}, "datastats": {}, "certify": {}, "efficiency": {}, "report": {},
    "fulltest": dict(n_full=None),                     # complete test split (7,180 PathMNIST images)
    "attacks": dict(n_attack=100),
    "ablation": dict(n_test=200),
    "gen_clip": dict(model="clip", n_full=None, **GEN_BUDGET),
    "gen_blood": dict(dataset="bloodmnist", n_full=None, **GEN_BUDGET),
}


def make_config(preset: str = "colab", stage: str = None, **overrides) -> Config:
    """defaults <- stage budget <- preset <- preset's stage budget <- explicit overrides."""
    cfg = Config()
    upd = {**STAGES.get(stage, {}), **PRESETS[preset], **PRESET_STAGES.get(preset, {}).get(stage, {}), **overrides}
    if preset == "smoke" and STAGES.get(stage, {}).get("n_full", 0) == 0:
        upd["n_full"] = 0
    for k, v in upd.items():
        if not hasattr(cfg, k):
            raise KeyError(f"unknown config key: {k}")
        setattr(cfg, k, v)
    return cfg


# Values reported by PromptSmooth (Hussein et al., MICCAI 2024), Table 2, PLIP on KatherColon:
# certified accuracy (%) at L2 radius, best over sigma in {0.1, 0.25, 0.5, 1.0}.
PROMPTSMOOTH_REPORTED_KATHER = {
    "radii": [0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5],
    "zeroshot_noPL": [49.4, 38.2, 20.8, 17.6, 11.0, 11.0, 11.0],
    "coop": [66.7, 56.0, 16.4, 14.2, 11.0, 11.0, 11.0],
    "denoised_smoothing": [48.2, 39.2, 31.0, 25.6, 17.4, 16.2, 14.6],
    "diffusion_smoothing": [57.0, 49.0, 41.0, 34.0, 26.0, 22.0, 16.0],
    "ps_zeroshot": [53.4, 49.0, 29.0, 29.0, 28.6, 28.4, 27.4],
    "ps_fewshot": [78.2, 67.6, 52.2, 35.6, 26.4, 22.2, 17.6],
    "ps_combined": [81.8, 81.0, 74.8, 73.2, 48.4, 47.2, 45.6],
}

In [ ]:
%%writefile coreprompt/utils.py
"""Seeding, checkpoint paths, timing and memory helpers."""
import json
import os
import random

import numpy as np
import torch


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ckpt_path(cfg, name: str) -> str:
    d = os.path.join(cfg.out_dir, "ckpt")
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, name)


def results_path(cfg, name: str) -> str:
    d = os.path.join(cfg.out_dir, "results")
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, name)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)


def load_json(path):
    with open(path) as f:
        return json.load(f)


def reset_peak_mem():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def peak_mem_mb():
    return torch.cuda.max_memory_allocated() / 2 ** 20 if torch.cuda.is_available() else 0.0


def get_device():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU visible to PyTorch. In Colab: Runtime > Change runtime type > T4 GPU, "
                           "then Runtime > Run all. (Running on CPU would take days.)")
    return "cuda"

In [ ]:
%%writefile coreprompt/data.py
"""Data loading for MedMNIST+ (Yang et al., Scientific Data 2023), sized for the Colab free tier.

PathMNIST is derived from the Kather colorectal histology data (NCT-CRC-HE-100K for train/val; CRC-VAL-HE-7K =
7,180 patches for test; Kather et al., PLoS Med 2019), i.e. the "KatherColon" benchmark used by PromptSmooth.
BloodMNIST (Acevedo et al., Data in Brief 2020; 3,421 test images) is used for cross-dataset generalisation.

Data budget (free tier)
  * The 224-px MedMNIST+ file of PathMNIST is ~12 GB, almost all of it the 90k-image training split, which few-shot
    prompt learning does not need. We read the .npz *remotely* with HTTP range requests and fetch only
      - the label arrays,
      - 16 images per class from the validation split (few-shot training set), and
      - the complete test split (7,180 PathMNIST / 3,421 BloodMNIST images, ~1.1 GB / ~0.5 GB as uint8).
    Download is roughly 1-2.5 GB in total depending on the file's compression, instead of 12 GB.
  * The complete test split is used for cheap full-test-set evaluations (clean and noisy accuracy, per-class
    accuracy); certification, attacks and ablations use the first N images of one fixed random permutation of it,
    so every smaller set is a subset of every larger one.
  * Everything is cached as uint8 .npz on Google Drive (OUT_DIR/data_cache); a reconnected session reads the cache.
  * If the server refuses range requests, the normal MedMNIST download is used as a fallback.
"""
import io
import os
import re
import time
import zipfile

import numpy as np
import torch

from medmnist import INFO

TEMPLATES = {
    "pathmnist": "an H&E image patch of {}.",          # PromptSmooth example: "An H&E image patch of {Tumor}"
    "bloodmnist": "a microscopy image of a {}.",
}
NOISY_INIT = {
    "pathmnist": "an H&E noisy image patch of",        # PromptSmooth Table 4: noisy context initialisation
    "bloodmnist": "a noisy microscopy image of a",
}
ZENODO = "https://zenodo.org/records/10519652/files/{flag}_{size}.npz?download=1"   # MedMNIST+ v3.0 record


def class_names(flag: str):
    labels = INFO[flag]["label"]
    names = [labels[str(i)] for i in range(len(labels))]
    return [re.sub(r"\s*\(.*?\)", "", n).strip() for n in names]   # drop parenthetical sub-types


# ----------------------------------------------------------------------------------------- remote .npz access
class HTTPRangeFile(io.RawIOBase):
    """Read-only, seekable file over HTTP range requests (lets zipfile/numpy read single arrays of a remote .npz)."""

    def __init__(self, url, min_chunk=1 << 18, max_chunk=1 << 25):
        import requests
        self.s = requests.Session()
        r = self.s.get(url, headers={"Range": "bytes=0-0"}, allow_redirects=True, timeout=60)
        if r.status_code != 206 or "Content-Range" not in r.headers:
            raise IOError(f"server does not support range requests (HTTP {r.status_code})")
        self.orig_url, self.url = url, r.url
        self.size = int(r.headers["Content-Range"].split("/")[-1])
        self.pos, self.cache_start, self.cache = 0, 0, b""
        self.min_chunk, self.max_chunk, self.chunk, self.last_end = min_chunk, max_chunk, min_chunk, -1
        self.downloaded = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def _fetch(self, start, n):
        end = min(self.size, start + n) - 1
        for attempt in range(8):
            try:
                r = self.s.get(self.url, headers={"Range": f"bytes={start}-{end}"}, timeout=180)
                if r.status_code == 206 and len(r.content) == end - start + 1:
                    self.downloaded += len(r.content)
                    return r.content
                if r.status_code in (401, 403, 404, 410):     # expired signed redirect: resolve the URL again
                    self.url = self.s.get(self.orig_url, headers={"Range": "bytes=0-0"}, timeout=60).url
            except Exception:
                pass
            time.sleep(min(60, 3 * 2 ** attempt))            # back-off (also for HTTP 429 rate limiting)
        raise IOError("range request failed")

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        cs, ce = self.cache_start, self.cache_start + len(self.cache)
        if not (cs <= self.pos and self.pos + n <= ce):
            # request size doubles while reading sequentially and resets on random access
            self.chunk = min(self.chunk * 2, self.max_chunk) if self.pos == self.last_end else self.min_chunk
            self.cache_start, self.cache = self.pos, self._fetch(self.pos, max(n, self.chunk))
            cs = self.cache_start
            self.last_end = cs + len(self.cache)
        out = self.cache[self.pos - cs:self.pos - cs + n]
        self.pos += len(out)
        return out

    def readinto(self, b):
        data = self.read(len(b))
        b[:len(data)] = data
        return len(data)


def _member(z, name):
    return z.open([m for m in z.namelist() if m.startswith(name)][0])


def _labels(z, split):
    with _member(z, f"{split}_labels") as m:
        return np.lib.format.read_array(m).reshape(-1).astype(np.int64)


def _rows(z, split, idx=None):
    """All rows (idx=None) or rows idx of {split}_images; seeks are ascending so compressed members stream once."""
    with _member(z, f"{split}_images") as m:
        version = np.lib.format.read_magic(m)
        shape, _, dtype = (np.lib.format.read_array_header_1_0(m) if version == (1, 0)
                           else np.lib.format.read_array_header_2_0(m))
        item = int(np.prod(shape[1:])) * dtype.itemsize
        if idx is None:
            buf = bytearray(shape[0] * item)
            mv, got, last = memoryview(buf), 0, 0
            print(f"[data] reading {split} split: {shape[0]} images ({len(buf) / 2 ** 30:.2f} GB uncompressed)",
                  flush=True)
            while got < len(buf):
                chunk = m.read(min(1 << 24, len(buf) - got))
                mv[got:got + len(chunk)] = chunk
                got += len(chunk)
                if got - last >= len(buf) // 10 or got == len(buf):
                    print(f"[data]   {split}: {100 * got / len(buf):.0f}%", flush=True)
                    last = got
            out = np.frombuffer(buf, dtype=dtype).reshape(shape)
        else:
            start = m.tell()
            out = np.empty((len(idx),) + tuple(shape[1:]), dtype=dtype)
            for k in np.argsort(idx):
                m.seek(start + int(idx[k]) * item)
                out[k] = np.frombuffer(m.read(item), dtype=dtype).reshape(shape[1:])
    if out.ndim == 3:                                   # grayscale -> RGB
        out = np.repeat(out[..., None], 3, axis=-1)
    return out


def stratified_indices(labels, per_class, rng):
    idx = []
    for c in np.unique(labels):
        pool = np.where(labels == c)[0]
        idx.extend(rng.choice(pool, size=min(per_class, len(pool)), replace=False).tolist())
    return np.array(sorted(idx))


def _sample(flag, size, shots, seed, url=None):
    """Few-shot images from val + complete test split, read over HTTP; falls back to the full MedMNIST download."""
    rng = np.random.default_rng(seed)
    try:
        f = HTTPRangeFile(url or ZENODO.format(flag=flag, size=size))
        z = zipfile.ZipFile(io.BufferedReader(f, buffer_size=1 << 16))
        print(f"[data] {flag}: reading labels and {shots}-shot images over HTTP ...", flush=True)
        y_val = _labels(z, "val")
        shot_idx = stratified_indices(y_val, shots, rng)
        x_shot = _rows(z, "val", shot_idx)
        y_te = _labels(z, "test")
        x_te = _rows(z, "test")
        print(f"[data] {flag}: read {len(shot_idx)} few-shot + {len(y_te)} test images; "
              f"downloaded {f.downloaded / 2 ** 30:.2f} GB")
    except Exception as e:                              # fallback: standard (full-file) MedMNIST download
        print(f"[data] remote read failed ({e}); using the standard MedMNIST download")
        import medmnist
        cls = getattr(medmnist, INFO[flag]["python_class"])
        dv = cls(split="val", download=True, size=size)
        y_val = dv.labels.reshape(-1).astype(np.int64)
        shot_idx = stratified_indices(y_val, shots, rng)
        x_shot = np.asarray(dv.imgs[shot_idx])
        del dv
        dt = cls(split="test", download=True, size=size)
        x_te, y_te = np.asarray(dt.imgs), dt.labels.reshape(-1).astype(np.int64)
        if x_shot.ndim == 3:
            x_shot, x_te = np.repeat(x_shot[..., None], 3, -1), np.repeat(x_te[..., None], 3, -1)
    perm = rng.permutation(len(y_te))
    return dict(x_shot=x_shot, y_shot=y_val[shot_idx], shot_idx=shot_idx, x_test=x_te, y_test=y_te, perm=perm,
                size=size)


def to_tensor01(imgs_uint8: np.ndarray) -> torch.Tensor:
    """uint8 (N,H,W,3) -> float32 (N,3,H,W) in [0,1] (CPU)."""
    return torch.from_numpy(np.ascontiguousarray(imgs_uint8)).permute(0, 3, 1, 2).contiguous().float().div_(255.0)


def iterate_full(data, batch, device):
    """Yields (x01 on device, y) over the complete test split without materialising it as float."""
    X, Y = data["x_full"], data["y_full"]
    for i in range(0, len(Y), batch):
        yield to_tensor01(X[i:i + batch]).to(device, non_blocking=True), torch.from_numpy(Y[i:i + batch])


HEAD = 2000   # small cache with the first HEAD images of the test permutation (fast loading for reduced budgets)


def _load_head(cfg, path, names):
    """Same images as the full cache (first rows of the fixed permutation), from a ~0.3 GB file instead of ~1.1 GB."""
    hp = path.replace(".npz", f"_head{HEAD}.npz")
    if not os.path.exists(hp):
        z = np.load(path)
        X, Yt, perm = z["x_test"], z["y_test"], z["perm"]
        h = perm[:HEAD]
        tmp = hp + ".tmp.npz"
        np.savez(tmp, x_head=X[h], y_head=Yt[h], perm_head=h, x_shot=z["x_shot"], y_shot=z["y_shot"],
                 shot_idx=z["shot_idx"], test_counts=np.bincount(Yt, minlength=len(names)), test_total=len(Yt),
                 size=z["size"])
        os.replace(tmp, hp)
        del X
    z = np.load(hp)
    Xh, Yh, ph = z["x_head"], z["y_head"], z["perm_head"]
    n, nf = cfg.n_test, cfg.n_full
    stats = dict(img_size=int(z["size"]), test_total=int(z["test_total"]), test_class_counts=z["test_counts"].tolist(),
                 shot_idx=z["shot_idx"].tolist(), test_idx=ph[:n].tolist())
    return dict(x_shot=to_tensor01(z["x_shot"]), y_shot=torch.from_numpy(z["y_shot"]),
                x_test=to_tensor01(Xh[:n]), y_test=torch.from_numpy(Yh[:n]),
                x_full=Xh[:nf] if nf > 0 else None, y_full=Yh[:nf] if nf > 0 else None, classnames=names,
                template=TEMPLATES[cfg.dataset], noisy_init=NOISY_INIT[cfg.dataset], stats=stats)


def load_data(cfg):
    cdir = cfg.data_dir or os.path.join(cfg.out_dir, "data_cache")
    os.makedirs(cdir, exist_ok=True)
    path = os.path.join(cdir, f"{cfg.dataset}_{cfg.img_size}_seed{cfg.seed}_shots{cfg.shots}.npz")
    if not os.path.exists(path):
        d = _sample(cfg.dataset, cfg.img_size, cfg.shots, cfg.seed, getattr(cfg, "data_url", None))
        tmp = path + ".tmp.npz"
        np.savez(tmp, **d)
        os.replace(tmp, path)
    names = class_names(cfg.dataset)
    if cfg.n_full is not None and max(cfg.n_test, cfg.n_full) <= HEAD:
        return _load_head(cfg, path, names)
    z = np.load(path)
    X, Yt = z["x_test"], z["y_test"]                 # each npz access re-reads the array: load once
    perm = z["perm"]
    full_n = len(perm) if cfg.n_full is None else min(cfg.n_full, len(perm))
    if full_n > 0:                       # complete test split only for stages that need it (saves RAM / Drive I/O)
        full_idx = np.sort(perm[:full_n])
        x_full, y_full = X[full_idx], Yt[full_idx]
    else:
        x_full, y_full = None, None
    sub = perm[:cfg.n_test]
    x_te, y_te = X[sub], Yt[sub]
    stats = dict(img_size=int(z["size"]), test_total=int(len(Yt)),
                 test_class_counts=np.bincount(Yt, minlength=len(names)).tolist(),
                 shot_idx=z["shot_idx"].tolist(), test_idx=sub.tolist())
    del X
    return dict(x_shot=to_tensor01(z["x_shot"]), y_shot=torch.from_numpy(z["y_shot"]),
                x_test=to_tensor01(x_te), y_test=torch.from_numpy(y_te),
                x_full=x_full, y_full=y_full, classnames=names, template=TEMPLATES[cfg.dataset],
                noisy_init=NOISY_INIT[cfg.dataset], stats=stats)

In [ ]:
%%writefile coreprompt/models.py
"""CLIP-style medical VLM wrapper (PLIP / CLIP) with CoOp-style learnable text context.

* PLIP: Huang et al., Nature Medicine 2023 (HF id ``vinid/plip``), a CLIP model fine-tuned on pathology image-text pairs.
* CLIP: Radford et al., ICML 2021 (HF id ``openai/clip-vit-base-patch32``), used for cross-model generalisation.
* Prompt learning follows CoOp (Zhou et al., IJCV 2022): M learnable context vectors [V]_1..[V]_M are prepended to the
  class-name tokens; the encoders stay frozen.

All image inputs are float tensors in [0,1] at native resolution. Normalisation (and resizing, if needed) is part of the
classifier, so Gaussian noise and adversarial perturbations live in [0,1] pixel space, as in Cohen et al. (2019).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

HF_IDS = {"plip": "vinid/plip", "clip": "openai/clip-vit-base-patch32"}


class MedVLM(nn.Module):
    def __init__(self, name: str = "plip", device: str = "cuda"):
        super().__init__()
        hf = HF_IDS[name]
        self.clip = CLIPModel.from_pretrained(hf, attn_implementation="eager").to(device).eval()
        self.processor = CLIPProcessor.from_pretrained(hf)
        self.tokenizer = self.processor.tokenizer
        for p in self.clip.parameters():
            p.requires_grad_(False)
        ip = self.processor.image_processor
        size = ip.crop_size["height"] if isinstance(ip.crop_size, dict) else ip.crop_size
        self.input_size = int(size)
        self.register_buffer("mean", torch.tensor(ip.image_mean, device=device).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(ip.image_std, device=device).view(1, 3, 1, 1))
        self.device = device
        self.name = name

    @property
    def logit_scale(self):
        return self.clip.logit_scale.exp().item()

    def encode_image(self, x01: torch.Tensor) -> torch.Tensor:
        """x01: (B,3,H,W) in [0,1] (noise may push values outside; no clipping, following Cohen et al.)."""
        if x01.shape[-1] != self.input_size:
            x01 = F.interpolate(x01, size=self.input_size, mode="bilinear", align_corners=False)
        x = (x01 - self.mean) / self.std
        with torch.autocast("cuda", dtype=torch.float16, enabled=(self.device == "cuda")):
            f = self.clip.get_image_features(pixel_values=x)
        if not torch.is_tensor(f):                       # some transformers versions return a model output
            f = getattr(f, "image_embeds", None) if getattr(f, "image_embeds", None) is not None else f.pooler_output
        return F.normalize(f.float(), dim=-1)

    @torch.no_grad()
    def encode_text_plain(self, prompts):
        tok = self.tokenizer(prompts, padding=True, truncation=True, max_length=77, return_tensors="pt").to(self.device)
        f = self.clip.get_text_features(**tok)
        if not torch.is_tensor(f):
            f = getattr(f, "text_embeds", None) if getattr(f, "text_embeds", None) is not None else f.pooler_output
        return F.normalize(f.float(), dim=-1)


class PromptLearner(nn.Module):
    """CoOp unified context: one set of M context vectors shared by all classes.

    Prompts are tokenised as ``"X X ... X {classname}."`` and the embeddings of the M placeholder tokens are replaced by
    the learnable context. Initialising the context from the embeddings of ``init_text`` reproduces the hand-crafted
    prompt ``"{init_text} {classname}."`` exactly (checked by ``check_equivalence``).
    """

    def __init__(self, vlm: MedVLM, classnames, init_text: str = None, n_ctx: int = None):
        super().__init__()
        tm = vlm.clip.text_model
        tok = vlm.tokenizer
        emb_layer = tm.embeddings.token_embedding
        if init_text is not None:
            init_ids = tok(init_text, add_special_tokens=False)["input_ids"]
            n_ctx = len(init_ids)
            with torch.no_grad():
                ctx = emb_layer(torch.tensor(init_ids, device=vlm.device)).clone()
        else:
            assert n_ctx is not None
            ctx = torch.empty(n_ctx, emb_layer.embedding_dim, device=vlm.device).normal_(std=0.02)
        self.n_ctx = n_ctx
        self.ctx = nn.Parameter(ctx.float())
        prompts = [" ".join(["X"] * n_ctx) + " " + c + "." for c in classnames]
        enc = tok(prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt")
        ids = enc["input_ids"].to(vlm.device)
        with torch.no_grad():
            emb = emb_layer(ids).float()
        self.register_buffer("prefix", emb[:, :1, :].clone())             # start-of-text token
        self.register_buffer("suffix", emb[:, 1 + n_ctx:, :].clone())     # class tokens, '.', EOS, padding
        eos_id = tok.eos_token_id
        self.register_buffer("eos_idx", (ids == eos_id).int().argmax(dim=-1))  # first EOS position
        self.vlm = vlm
        L = ids.shape[1]
        mask = torch.full((L, L), float("-inf"), device=vlm.device).triu_(1)
        self.register_buffer("causal_mask", mask[None, None])
        self.register_buffer("pos_ids", torch.arange(L, device=vlm.device)[None])

    def text_features(self, ctx: torch.Tensor = None) -> torch.Tensor:
        """Returns L2-normalised class embeddings (C, D); differentiable w.r.t. ctx."""
        ctx = self.ctx if ctx is None else ctx
        C = self.prefix.shape[0]
        tm = self.vlm.clip.text_model
        x = torch.cat([self.prefix, ctx.unsqueeze(0).expand(C, -1, -1), self.suffix], dim=1)
        h = x + tm.embeddings.position_embedding(self.pos_ids)
        out = _run_encoder(self, h)
        out = tm.final_layer_norm(out)
        pooled = out[torch.arange(C, device=out.device), self.eos_idx]
        return F.normalize(self.vlm.clip.text_projection(pooled), dim=-1)


# How the causal mask is handed to the CLIP text encoder differs across transformers versions. The first call
# calibrates the calling convention against the official text encoder and caches it for the process.
_ENCODER_MODE = {"mode": None}
_MODES = ("causal_kw", "attn_kw")


def _encode_with(mode, pl, h):
    enc = pl.vlm.clip.text_model.encoder
    L = h.shape[1]
    # eager CLIPAttention requires the causal mask to be exactly (batch, 1, L, L): no broadcasting
    m = pl.causal_mask.to(h.dtype).expand(h.shape[0], 1, L, L).contiguous()
    out = enc(inputs_embeds=h, causal_attention_mask=m) if mode == "causal_kw" else enc(inputs_embeds=h, attention_mask=m)
    return out[0] if isinstance(out, (tuple, list)) else out.last_hidden_state


def _run_encoder(pl, h):
    if _ENCODER_MODE["mode"] is None:
        _calibrate(pl.vlm)
    return _encode_with(_ENCODER_MODE["mode"], pl, h)


@torch.no_grad()
def _calibrate(vlm, probe=("a photo of", ["dog", "smooth muscle"])):
    init, names = probe
    ref = vlm.encode_text_plain([f"{init} {c}." for c in names])
    errors = {}
    for mode in _MODES:
        _ENCODER_MODE["mode"] = mode
        try:
            pl = PromptLearner(vlm, names, init_text=init)
            cos = (pl.text_features() * ref).sum(-1).min().item()
            if cos > 0.999:
                return mode
            errors[mode] = f"cosine {cos:.4f}"
        except Exception as e:          # calling convention not supported by this transformers version
            errors[mode] = repr(e)
    _ENCODER_MODE["mode"] = None
    raise RuntimeError(f"could not reproduce the CLIP text encoder with learnable context: {errors}")


@torch.no_grad()
def check_equivalence(vlm: MedVLM, classnames, init_text: str):
    """Sanity check: learnable-context encoder initialised from init_text == official text encoder on the same prompt."""
    pl = PromptLearner(vlm, classnames, init_text=init_text)
    ours = pl.text_features()
    ref = vlm.encode_text_plain([f"{init_text} {c}." for c in classnames])
    cos = (ours * ref).sum(-1)
    return cos.min().item()


def logits_from(vlm: MedVLM, img_feats: torch.Tensor, txt_feats: torch.Tensor) -> torch.Tensor:
    return vlm.logit_scale * img_feats @ txt_feats.t()


class Classifier(nn.Module):
    """Base classifier f(x) = argmax_c <E_img(x), E_txt(t_c)>; text features are fixed (precomputed)."""

    def __init__(self, vlm: MedVLM, txt_feats: torch.Tensor):
        super().__init__()
        self.vlm = vlm
        self.register_buffer("txt", txt_feats.detach())

    def forward(self, x01):
        return logits_from(self.vlm, self.vlm.encode_image(x01), self.txt)

In [ ]:
%%writefile coreprompt/losses.py
"""Training objectives.

Notation: for an input x with label y, m Gaussian draws delta_i ~ N(0, sigma^2 I), base-classifier logits
z_i = s * <E_img(x + delta_i), T(P)>, probabilities p_i = softmax(z_i), and the empirical smoothed distribution
p_bar = (1/m) sum_i p_i.

* ce_noise      - Gaussian data augmentation (Cohen et al. 2019); the few-shot PromptSmooth objective.
* consistency   - Jeong & Shin (NeurIPS 2020): their L_con = KL(p_bar || p_i) + H(p_bar) weighted by lambda; here the
                  two parts get separate weights, lambda * mean_i KL(p_bar || p_i) + eta * H(p_bar) (eta = lambda recovers it).
* radius_hinge  - MACER (Zhai et al., ICLR 2020): (sigma/2) * (1/B) * sum over samples the soft smoothed classifier
                  gets right of max(gamma - xi, 0), xi = Phi^-1(p_hat_y) - Phi^-1(max_{c!=y} p_hat_c),
                  p_hat = mean_i softmax(beta * z_i); normalised by the full batch size B as in MACER's Algorithm 1.
"""
import torch
import torch.nn.functional as F



def ce_noise(logits_m, y):
    """logits_m: (m, B, C)."""
    m, B, C = logits_m.shape
    return F.cross_entropy(logits_m.reshape(m * B, C), y.repeat(m))


def consistency(logits_m, lam: float, eta: float, eps: float = 1e-8):
    p = logits_m.softmax(-1)                     # (m,B,C)
    p_bar = p.mean(0)                            # (B,C)
    kl = (p_bar.unsqueeze(0) * (torch.log(p_bar.unsqueeze(0) + eps) - torch.log(p + eps))).sum(-1).mean()
    ent = -(p_bar * torch.log(p_bar + eps)).sum(-1).mean()
    return lam * kl + eta * ent, kl.detach(), ent.detach()


def radius_hinge(logits_m, y, sigma: float, gamma: float, beta: float, clamp: float = 1e-4):
    p_hat = (beta * logits_m).softmax(-1).mean(0).clamp(clamp, 1 - clamp)   # (B,C)
    top2 = p_hat.topk(2, dim=-1)
    correct = top2.indices[:, 0] == y
    if correct.sum() == 0:
        return logits_m.sum() * 0.0, torch.tensor(0.0, device=logits_m.device)
    p_y = p_hat.gather(1, y[:, None]).squeeze(1)
    p_other = p_hat.scatter(1, y[:, None], 0.0).max(-1).values
    xi = torch.special.ndtri(p_y) - torch.special.ndtri(p_other)          # soft certified-margin in units of sigma/2
    hinge = F.relu(gamma - xi)[correct]
    radius = (sigma / 2.0) * xi[correct]
    return (sigma / 2.0) * hinge.mean() * correct.float().mean(), radius.mean().detach()


def entropy_of_mean(logits_m, eps: float = 1e-8):
    """Test-time objective of zero-shot PromptSmooth: entropy of the averaged prediction over noisy copies."""
    p_bar = logits_m.softmax(-1).mean(0)
    return -(p_bar * torch.log(p_bar + eps)).sum(-1).mean()

In [ ]:
%%writefile coreprompt/train.py
"""Few-shot prompt learning: CoOp, PromptSmooth (few-shot) and the proposed CoRe-Prompt.

Only the M context vectors are trained (M x 512 parameters); both encoders are frozen, so image features of noisy
inputs are computed without gradients and the backward pass runs through the text encoder only.
"""
import math
import os
import time

import torch

from .losses import ce_noise, consistency, radius_hinge
from .models import PromptLearner, logits_from
from .utils import ckpt_path, peak_mem_mb, reset_peak_mem, set_seed

METHODS = ("coop", "ps_fewshot", "coreprompt")


def train_prompt(vlm, data, cfg, method: str, sigma: float, use_con: bool = True, use_rad: bool = True,
                 init: str = "random", n_ctx: int = 5, tag: str = "", verbose: bool = True):
    """Returns (ctx tensor, log dict). Results are cached under cfg.out_dir/ckpt."""
    assert method in METHODS
    name = f"{cfg.dataset}_{cfg.model}_{method}{tag}_s{sigma}_seed{cfg.seed}_shots{cfg.shots}"
    path = ckpt_path(cfg, name + ".pt")
    if os.path.exists(path):
        blob = torch.load(path, map_location=vlm.device)
        if verbose:
            print(f"[ckpt] loaded {name}")
        return blob["ctx"], blob["log"]

    set_seed(cfg.seed)
    init_text = data["noisy_init"] if init == "noisy_text" else (
        data["template"].replace(" {}.", "") if init == "template" else None)
    pl = PromptLearner(vlm, data["classnames"], init_text=init_text, n_ctx=None if init_text else n_ctx)
    opt = torch.optim.SGD([pl.ctx], lr=cfg.lr, momentum=0.9)
    x_all, y_all = data["x_shot"], data["y_shot"]
    steps_per_epoch = math.ceil(len(x_all) / cfg.batch)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs * steps_per_epoch)
    noisy = method != "coop"
    m = cfg.m_train if noisy else 1
    log = dict(method=method, sigma=sigma, use_con=use_con, use_rad=use_rad, init=init, n_ctx=pl.n_ctx,
               trainable_params=pl.ctx.numel(), epoch_loss=[], soft_radius=[])
    reset_peak_mem()
    t0 = time.time()
    g = torch.Generator().manual_seed(cfg.seed)
    for ep in range(cfg.epochs):
        perm = torch.randperm(len(x_all), generator=g)
        tot = 0.0
        rad_ep = []
        for s in range(steps_per_epoch):
            idx = perm[s * cfg.batch:(s + 1) * cfg.batch]
            x = x_all[idx].to(vlm.device, non_blocking=True)
            y = y_all[idx].to(vlm.device)
            with torch.no_grad():
                xs = x.unsqueeze(0).repeat(m, 1, 1, 1, 1)                      # (m,B,3,H,W)
                if noisy:
                    xs = xs + sigma * torch.randn_like(xs)
                f = vlm.encode_image(xs.flatten(0, 1)).view(m, len(idx), -1)  # (m,B,D)
            t = pl.text_features()
            z = logits_from(vlm, f, t)                                       # (m,B,C)
            loss = ce_noise(z, y)
            if method == "coreprompt":
                if use_con:
                    lc, _, _ = consistency(z, cfg.lam_con, cfg.eta_ent)
                    loss = loss + lc
                if use_rad:
                    lr_, r = radius_hinge(z, y, sigma, cfg.gamma_rad, cfg.beta_rad)
                    loss = loss + cfg.mu_rad * lr_
                    rad_ep.append(r.item())
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item()
        log["epoch_loss"].append(tot / steps_per_epoch)
        log["soft_radius"].append(sum(rad_ep) / len(rad_ep) if rad_ep else None)
        if verbose and (ep % 10 == 0 or ep == cfg.epochs - 1):
            print(f"  [{method} s={sigma}] epoch {ep + 1}/{cfg.epochs} loss={log['epoch_loss'][-1]:.4f}")
    log["train_time_s"] = time.time() - t0
    log["peak_mem_mb"] = peak_mem_mb()
    ctx = pl.ctx.detach().clone()
    torch.save(dict(ctx=ctx, log=log), path)
    return ctx, log


def text_feats_for(vlm, data, ctx):
    pl = PromptLearner(vlm, data["classnames"], n_ctx=ctx.shape[0])
    with torch.no_grad():
        return pl.text_features(ctx.to(vlm.device))


def zeroshot_text_feats(vlm, data):
    return vlm.encode_text_plain([data["template"].format(c) for c in data["classnames"]])

In [ ]:
%%writefile coreprompt/tta.py
"""Zero-shot PromptSmooth (Hussein et al., MICCAI 2024, Sec. 2.3): per-image test-time prompt tuning.

For each test image x, T Gaussian-noisy copies are drawn and the context vectors are updated with `steps` gradient
step(s) to minimise the entropy of the averaged prediction (TPT-style, Shu et al., NeurIPS 2022). The adapted prompt
is then used by CERTIFY for that image only. Starting from the hand-crafted prompt gives "PromptSmooth zero-shot";
starting from the few-shot-learned prompt gives "PromptSmooth (combined)". Applying the same step on top of a
CoRe-Prompt context gives "CoRe-Prompt + TTA".

The adaptation noise is drawn independently of the certification noise. As in PromptSmooth, the certificate is
issued for the smoothed classifier with the adapted prompt held fixed; because the prompt itself depends on x, this is
not an end-to-end guarantee for the input-dependent pipeline (an attacker moving x also moves the prompt). The paper
reports TTA variants with this caveat and treats the non-TTA CoRe-Prompt as the method with a strict certificate.
"""
import torch

from .losses import entropy_of_mean
from .models import PromptLearner, logits_from


class TestTimeAdapter:
    def __init__(self, vlm, classnames, ctx_init: torch.Tensor, sigma: float, T: int, steps: int, lr: float):
        self.vlm = vlm
        self.pl = PromptLearner(vlm, classnames, n_ctx=ctx_init.shape[0])
        self.ctx0 = ctx_init.detach().to(vlm.device).float()
        self.sigma, self.T, self.steps, self.lr = sigma, T, steps, lr

    def __call__(self, x01: torch.Tensor) -> torch.Tensor:
        """Returns adapted, normalised text features (C,D) for a single image (3,H,W)."""
        ctx = self.ctx0.clone().requires_grad_(True)
        opt = torch.optim.AdamW([ctx], lr=self.lr)   # TPT uses AdamW
        with torch.no_grad():
            xs = x01.unsqueeze(0).expand(self.T, -1, -1, -1)
            f = self.vlm.encode_image(xs + self.sigma * torch.randn_like(xs))      # (T,D)
        for _ in range(self.steps):
            t = self.pl.text_features(ctx)
            loss = entropy_of_mean(logits_from(self.vlm, f, t).unsqueeze(1))      # (T,1,C)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        with torch.no_grad():
            return self.pl.text_features(ctx.detach())

In [ ]:
%%writefile coreprompt/methods.py
"""Factory that turns a method name into text features (or a per-image test-time adapter) for a given sigma.

Method names
  zs_plip        hand-crafted prompt, no training ("Zero-shot (No PL)" in PromptSmooth Table 2)
  coop           CoOp trained on clean images (PromptSmooth's CoOp baseline)
  ps_fewshot     PromptSmooth few-shot: prompt trained on Gaussian-noisy images (CE)
  ps_zeroshot    PromptSmooth zero-shot: test-time entropy adaptation from the hand-crafted noisy prompt
  ps_combined    PromptSmooth combined: test-time adaptation from the few-shot noisy prompt
  coreprompt     proposed: CE + consistency + radius hinge (Eq. 9)
  coreprompt_tta proposed + PromptSmooth-style test-time adaptation
  core_con / core_rad  ablations (consistency only / radius only)
"""
import torch

from .models import PromptLearner
from .train import text_feats_for, train_prompt, zeroshot_text_feats
from .tta import TestTimeAdapter

ALL_METHODS = ["zs_plip", "coop", "ps_fewshot", "ps_zeroshot", "ps_combined", "coreprompt", "coreprompt_tta"]


def _init_ctx(vlm, data):
    with torch.no_grad():
        return PromptLearner(vlm, data["classnames"], init_text=data["noisy_init"]).ctx.detach().clone()


def build(vlm, data, cfg, sigma, names, logs=None, tag="", **train_kw):
    out = {}
    logs = {} if logs is None else logs
    cache = {}

    def trained(method, sub="", **kw):
        key = (method, sub)
        if key not in cache:
            init = "template" if method == "coop" else "noisy_text"
            ctx, log = train_prompt(vlm, data, cfg, method, sigma, init=init, tag=tag + sub, **{**train_kw, **kw})
            cache[key] = ctx
            logs[method + sub] = log
        return cache[key]

    def tta(ctx):
        return TestTimeAdapter(vlm, data["classnames"], ctx, sigma, cfg.tta_T, cfg.tta_steps, cfg.tta_lr)

    for name in names:
        if name == "zs_plip":
            out[name] = zeroshot_text_feats(vlm, data)
        elif name == "coop":
            out[name] = text_feats_for(vlm, data, trained("coop"))
        elif name == "ps_fewshot":
            out[name] = text_feats_for(vlm, data, trained("ps_fewshot"))
        elif name == "ps_zeroshot":
            out[name] = tta(_init_ctx(vlm, data))
        elif name == "ps_combined":
            out[name] = tta(trained("ps_fewshot"))
        elif name == "coreprompt":
            out[name] = text_feats_for(vlm, data, trained("coreprompt"))
        elif name == "coreprompt_tta":
            out[name] = tta(trained("coreprompt"))
        elif name == "core_con":
            out[name] = text_feats_for(vlm, data, trained("coreprompt", sub="_con", use_rad=False))
        elif name == "core_rad":
            out[name] = text_feats_for(vlm, data, trained("coreprompt", sub="_rad", use_con=False))
        else:
            raise KeyError(name)
    return out

In [ ]:
%%writefile coreprompt/smoothing.py
"""Randomized smoothing (Cohen, Rosenfeld & Kolter, ICML 2019).

Smoothed classifier  g(x) = argmax_c P_{delta ~ N(0, sigma^2 I)} [ f(x + delta) = c ].
CERTIFY: choose c_A with n0 samples, estimate p_A with n samples, lower-bound it with a one-sided Clopper-Pearson
interval at level 1-alpha; if pA_lower > 1/2 return (c_A, R = sigma * Phi^-1(pA_lower)), otherwise ABSTAIN.

Efficiency (this work): all compared methods share the frozen image encoder and differ only in their text features
T_k (C x D). For each test image and noise draw we compute the image embedding once and score it against every T_k.
The certification cost therefore grows with the number of image-encoder passes (n0 + n per image), not with the number
of methods, and all methods are certified with the *same* noise samples (a paired comparison).

Checkpointing: per-image results are appended to a JSONL file, so an interrupted Colab session resumes where it stopped.
"""
import json
import os
import time

import numpy as np
import torch
from scipy.stats import beta, norm

ABSTAIN = -1


def clopper_pearson_lower(k: int, n: int, alpha: float) -> float:
    """One-sided (1-alpha) lower confidence bound for a binomial proportion = two-sided CP at 2*alpha (Cohen et al.)."""
    return 0.0 if k == 0 else float(beta.ppf(alpha, k, n - k + 1))


def certify_from_counts(counts0: np.ndarray, counts: np.ndarray, n: int, alpha: float, sigma: float):
    c_a = int(np.argmax(counts0))
    p_lower = clopper_pearson_lower(int(counts[c_a]), n, alpha)
    if p_lower < 0.5:
        return ABSTAIN, 0.0, p_lower
    return c_a, float(sigma * norm.ppf(p_lower)), p_lower


def max_certifiable_radius(n: int, alpha: float, sigma: float) -> float:
    return float(sigma * norm.ppf(alpha ** (1.0 / n)))


@torch.no_grad()
def noisy_feature_counts(vlm, x01, txt_list, sigma, num, batch):
    """Sample `num` noisy copies of a single image, return class-count arrays for each text-feature set."""
    C = txt_list[0].shape[0]
    counts = [np.zeros(C, dtype=np.int64) for _ in txt_list]
    remaining = num
    while remaining > 0:
        b = min(batch, remaining)
        xs = x01.unsqueeze(0).expand(b, -1, -1, -1)
        f = vlm.encode_image(xs + sigma * torch.randn_like(xs))
        for k, t in enumerate(txt_list):
            pred = (f @ t.t()).argmax(-1).cpu().numpy()
            counts[k] += np.bincount(pred, minlength=C)
        remaining -= b
    return counts


def certify_methods(vlm, x_test, y_test, methods: dict, sigma: float, cfg, out_jsonl: str, verbose=True):
    """methods: {name: text_features (C,D)} or {name: callable(x01)->text_features} for per-image adapted prompts
    (zero-shot PromptSmooth). Writes one JSON line per (image, method). Resumes per (image, method): methods added
    in a later run are certified for every image, all methods of one call share the same noise samples."""
    done = set()
    if os.path.exists(out_jsonl):
        with open(out_jsonl) as f:
            for line in f:
                r = json.loads(line)
                done.add((r["idx"], r["method"]))
    names = list(methods.keys())
    t_start = time.time()
    n_new = 0
    with open(out_jsonl, "a") as fout:
        for i in range(len(y_test)):
            todo = [k for k in names if (i, k) not in done]      # resume per (image, method)
            if not todo:
                continue
            x = x_test[i].to(vlm.device)
            y = int(y_test[i])
            t0 = time.time()
            txt = [methods[k](x) if callable(methods[k]) else methods[k] for k in todo]
            c0 = noisy_feature_counts(vlm, x, txt, sigma, cfg.n0, cfg.cert_batch)
            c1 = noisy_feature_counts(vlm, x, txt, sigma, cfg.n, cfg.cert_batch)
            dt = time.time() - t0
            for k, name in enumerate(todo):
                pred, rad, pl = certify_from_counts(c0[k], c1[k], cfg.n, cfg.alpha, sigma)
                fout.write(json.dumps(dict(idx=i, method=name, sigma=sigma, label=y, predict=pred, radius=rad,
                                           pA_lower=pl, correct=int(pred == y), time_s=dt)) + "\n")
            fout.flush()
            n_new += 1
            if verbose and (i % 50 == 0):
                el = time.time() - t_start
                print(f"  sigma={sigma} img {i + 1}/{len(y_test)} ({el / max(n_new, 1):.1f}s/img)")
    return out_jsonl


def load_cert(jsonl_paths):
    import pandas as pd
    rows = []
    for p in ([jsonl_paths] if isinstance(jsonl_paths, str) else jsonl_paths):
        with open(p) as f:
            rows += [json.loads(l) for l in f]
    return pd.DataFrame(rows)


def certified_accuracy(df, radii):
    """Certified accuracy at radius r = fraction of test points with correct prediction and certified radius >= r.
    For r = 0 this is the clean accuracy of the smoothed classifier (abstentions count as errors)."""
    out = {}
    for r in radii:
        out[r] = float(((df["correct"] == 1) & (df["radius"] >= r)).mean())
    return out


def envelope(df, radii):
    """Best certified accuracy over sigma at each radius (Cohen et al. 2019; PromptSmooth Table 2 convention)."""
    best = {r: 0.0 for r in radii}
    for s, g in df.groupby("sigma"):
        ca = certified_accuracy(g, radii)
        for r in radii:
            best[r] = max(best[r], ca[r])
    return best


def acr(df):
    """Average certified radius (Zhai et al. 2020): mean over test points of R * 1[correct] (abstain -> 0)."""
    return float((df["radius"] * df["correct"]).mean())

In [ ]:
%%writefile coreprompt/defenses.py
"""Input-transformation defences used in the adaptive-evaluation study.

* bit_depth   - feature squeezing by colour-depth reduction (Xu, Evans & Qi, NDSS 2018); non-differentiable (round).
* jpeg        - JPEG re-compression (Guo et al., ICLR 2018; Dziugaite et al. 2016); non-differentiable.
* resize_pad  - random resizing and padding (Xie et al., ICLR 2018); randomised.

Each defence exposes forward(x) and a BPDA backward approximation g(x) = x (Athalye, Carlini & Wagner, ICML 2018),
which is the standard straight-through approximation for these transforms. Randomised defences are attacked with
Expectation over Transformation (EOT).
"""
import io

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image


def bit_depth(x, bits: int = 4):
    levels = 2 ** bits - 1
    return torch.round(x.clamp(0, 1) * levels) / levels


def jpeg(x, quality: int = 75):
    out = []
    for img in x.detach().clamp(0, 1).cpu():
        arr = (img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
        buf = io.BytesIO()
        Image.fromarray(arr).save(buf, format="JPEG", quality=quality)
        rec = np.asarray(Image.open(io.BytesIO(buf.getvalue())).convert("RGB"), dtype=np.float32) / 255.0
        out.append(torch.from_numpy(rec).permute(2, 0, 1))
    return torch.stack(out).to(x.device)


def resize_pad(x, max_ratio: float = 1.12):
    """Xie et al.: random rescale to [S, S*max_ratio) then random zero-pad to the max size; then resize back to S."""
    S = x.shape[-1]
    big = int(round(S * max_ratio))
    r = int(torch.randint(S, big, (1,)).item())
    y = F.interpolate(x, size=r, mode="nearest")
    pl = int(torch.randint(0, big - r + 1, (1,)).item())
    pt = int(torch.randint(0, big - r + 1, (1,)).item())
    y = F.pad(y, (pl, big - r - pl, pt, big - r - pt))
    return F.interpolate(y, size=S, mode="bilinear", align_corners=False)


class _BPDA(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, fn):
        return fn(x)

    @staticmethod
    def backward(ctx, g):
        return g, None


DEFENCES = {
    "none": dict(fn=lambda x: x, random=False, differentiable=True),
    "bit_depth": dict(fn=bit_depth, random=False, differentiable=False),
    "jpeg": dict(fn=jpeg, random=False, differentiable=False),
    "resize_pad": dict(fn=resize_pad, random=True, differentiable=True),
}


def apply(name, x):
    """Forward = the real defence; backward = exact gradient if differentiable, otherwise BPDA identity."""
    d = DEFENCES[name]
    return d["fn"](x) if d["differentiable"] else _BPDA.apply(x, d["fn"])

In [ ]:
%%writefile coreprompt/attacks.py
"""White-box attacks for empirical robustness evaluation.

* PGD-Linf / PGD-L2 (Madry et al., ICLR 2018) on the base classifier, optionally through a defence
  (BPDA for non-differentiable transforms, EOT for randomised ones; Athalye et al. 2018).
* SmoothAdv attack on the smoothed classifier (Salman et al., NeurIPS 2019): PGD-L2 on the soft smoothed classifier,
  maximising  -log( (1/m) sum_j softmax(f(x' + delta_j))_y ),  delta_j ~ N(0, sigma^2 I) re-drawn every step.
* PREDICT (Cohen et al. 2019) for evaluating the smoothed classifier on (adversarial) inputs, with abstention.

All perturbations are in [0,1] pixel space at the dataset resolution, and adversarial images are clipped to [0,1].
Following Carlini et al. (2019) the gradient of every attack is accumulated over EOT draws *before* the step.
"""
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import binomtest

from . import defenses
from .models import logits_from

ABSTAIN = -1


def base_logits_fn(vlm, txt, defence: str = "none"):
    def fn(x):
        return logits_from(vlm, vlm.encode_image(defenses.apply(defence, x)), txt)
    return fn


def _l2_normalise(g):
    n = g.flatten(1).norm(dim=1).clamp_min(1e-12).view(-1, 1, 1, 1)
    return g / n


def _project_l2(delta, eps):
    n = delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1)
    return delta * torch.clamp(eps / n.clamp_min(1e-12), max=1.0)


def pgd(logits_fn, x, y, eps, steps, norm="linf", eot=1, step_size=None, rand_start=True, noise_sigma=0.0):
    """Generic PGD. If noise_sigma > 0 the loss is the SmoothAdv soft-smoothed loss over `eot` Gaussian draws;
    otherwise the per-draw CE is averaged over `eot` evaluations (EOT for randomised defences)."""
    x = x.detach()
    if step_size is None:
        step_size = 2.5 * eps / steps          # common PGD heuristic (e.g. robustness library)
    if rand_start:
        if norm == "linf":
            delta = torch.empty_like(x).uniform_(-eps, eps)
        else:
            delta = _project_l2(torch.randn_like(x), eps * torch.rand(len(x), 1, 1, 1, device=x.device))
    else:
        delta = torch.zeros_like(x)
    delta = ((x + delta).clamp(0, 1) - x).detach()
    for _ in range(steps):
        delta.requires_grad_(True)
        xa = x + delta
        if noise_sigma > 0:
            probs = 0
            for _ in range(eot):
                probs = probs + logits_fn(xa + noise_sigma * torch.randn_like(xa)).softmax(-1)
            loss = F.nll_loss(torch.log(probs / eot + 1e-12), y)
        else:
            loss = 0
            for _ in range(eot):
                loss = loss + F.cross_entropy(logits_fn(xa), y)
            loss = loss / eot
        g, = torch.autograd.grad(loss, delta)
        with torch.no_grad():
            if norm == "linf":
                delta = (delta + step_size * g.sign()).clamp(-eps, eps)
            else:
                delta = _project_l2(delta + step_size * _l2_normalise(g), eps)
            delta = ((x + delta).clamp(0, 1) - x)
        delta = delta.detach()
    return (x + delta).detach()


@torch.no_grad()
def predict_smoothed(vlm, txt, x01, sigma, n, alpha, batch=250):
    """Cohen et al. PREDICT for one image: returns class or ABSTAIN (two-sided binomial test on top-2 counts)."""
    C = txt.shape[0]
    counts = np.zeros(C, dtype=np.int64)
    rem = n
    while rem > 0:
        b = min(batch, rem)
        xs = x01.unsqueeze(0).expand(b, -1, -1, -1)
        pred = (vlm.encode_image(xs + sigma * torch.randn_like(xs)) @ txt.t()).argmax(-1).cpu().numpy()
        counts += np.bincount(pred, minlength=C)
        rem -= b
    top2 = counts.argsort()[::-1][:2]
    na, nb = int(counts[top2[0]]), int(counts[top2[1]])
    if binomtest(na, na + nb, 0.5).pvalue > alpha:
        return ABSTAIN
    return int(top2[0])


@torch.no_grad()
def base_accuracy(logits_fn, x, y, repeats=1):
    """Accuracy of a (possibly randomised) defended base classifier; randomised defences use majority over repeats."""
    votes = torch.stack([logits_fn(x).argmax(-1) for _ in range(repeats)])
    pred = torch.mode(votes, dim=0).values
    return (pred == y).float()

In [ ]:
%%writefile coreprompt/report.py
"""Tables (CSV + LaTeX) and figures for Section 4, built from the raw results of every stage.

Pure numpy / pandas / matplotlib / scipy (no torch), so it can be re-run anywhere on a copy of OUT_DIR/results.
Every section is optional: missing results are skipped with a note, so the report also works on partial runs.
Output: OUT_DIR/report/{figures,tables}/ and OUT_DIR/report/index.json (ordered list used by the notebook).
"""
import glob
import json
import os
import traceback

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from scipy.stats import binomtest  # noqa: E402

REPORTED = {   # PromptSmooth (Hussein et al., MICCAI 2024) Table 2, PLIP / KatherColon, best over sigma
    "radii": [0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5],
    "zs_plip": [49.4, 38.2, 20.8, 17.6, 11.0, 11.0, 11.0],
    "coop": [66.7, 56.0, 16.4, 14.2, 11.0, 11.0, 11.0],
    "ps_zeroshot": [53.4, 49.0, 29.0, 29.0, 28.6, 28.4, 27.4],
    "ps_fewshot": [78.2, 67.6, 52.2, 35.6, 26.4, 22.2, 17.6],
    "ps_combined": [81.8, 81.0, 74.8, 73.2, 48.4, 47.2, 45.6],
    "denoised_smoothing": [48.2, 39.2, 31.0, 25.6, 17.4, 16.2, 14.6],
    "diffusion_smoothing": [57.0, 49.0, 41.0, 34.0, 26.0, 22.0, 16.0],
}
NAMES = {"zs_plip": "Zero-shot + RS", "coop": "CoOp + RS", "ps_zeroshot": "PS zero-shot†",
         "ps_fewshot": "PS few-shot", "ps_combined": "PS combined†", "coreprompt": "CoRe-Prompt (ours)",
         "coreprompt_tta": "CoRe-Prompt + TTA† (ours)", "denoised_smoothing": "Denoised smoothing (reported)",
         "diffusion_smoothing": "Diffusion smoothing (reported)"}
ORDER = ["zs_plip", "coop", "ps_zeroshot", "ps_fewshot", "ps_combined", "coreprompt", "coreprompt_tta"]
COLORS = {"zs_plip": "#999999", "coop": "#56B4E9", "ps_zeroshot": "#F0E442", "ps_fewshot": "#E69F00",
          "ps_combined": "#D55E00", "coreprompt": "#0072B2", "coreprompt_tta": "#009E73",
          "denoised_smoothing": "#666666", "diffusion_smoothing": "#333333"}
STYLE = {k: ("--" if k in ("ps_zeroshot", "ps_combined", "coreprompt_tta") else "-") for k in COLORS}
RADII = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
ABL = {"abl_ce": "$L_{CE}$ (= PS few-shot)", "abl_con": "$L_{CE}+L_{con}$", "abl_rad": "$L_{CE}+\\mu L_{rad}$",
       "abl_full": "Full CoRe-Prompt"}
DEFAULTS = {"lam_con": 10.0, "eta_ent": 0.5, "mu_rad": 1.0, "gamma_rad": 4.0, "beta_rad": 1.0, "m_train": 4,
            "shots": 16, "n_ctx_random": None}
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})


# ----------------------------------------------------------------------------------------------- helpers
class Report:
    def __init__(self, out_dir):
        self.res = os.path.join(out_dir, "results")
        self.dir = os.path.join(out_dir, "report")
        self.fig_dir = os.path.join(self.dir, "figures")
        self.tab_dir = os.path.join(self.dir, "tables")
        os.makedirs(self.fig_dir, exist_ok=True)
        os.makedirs(self.tab_dir, exist_ok=True)
        self.index = []

    def add(self, section, title, fig=None, table=None, note=None):
        self.index.append(dict(section=section, title=title, figure=fig, table=table, note=note))

    def savefig(self, fig, name):
        fig.tight_layout()
        fig.savefig(os.path.join(self.fig_dir, name + ".png"), dpi=160, bbox_inches="tight")
        fig.savefig(os.path.join(self.fig_dir, name + ".pdf"), bbox_inches="tight")
        plt.close(fig)
        return os.path.join("figures", name + ".png")

    def savetab(self, df, name, caption, label, digits=1):
        df.to_csv(os.path.join(self.tab_dir, name + ".csv"), index=False)
        with open(os.path.join(self.tab_dir, name + ".tex"), "w") as f:
            f.write(to_latex(df, caption, label, digits))
        return os.path.join("tables", name + ".csv")


def to_latex(df, caption, label, digits=1):
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return "--" if np.isnan(v) else f"{v:.{digits}f}"
        return str(v).replace("%", "\\%").replace("&", "\\&").replace("_", "\\_").replace("†", "$^\\dagger$")
    cols = " & ".join(fmt(c) for c in df.columns)
    rows = "\n".join(" & ".join(fmt(v) for v in r) + " \\\\" for r in df.itertuples(index=False))
    return ("\\begin{table}[!t]\n\\caption{" + caption + "}\\label{" + label + "}\n\\footnotesize\n"
            "\\begin{tabular}{@{}" + "l" * df.shape[1] + "@{}}\n\\toprule\n" + cols + " \\\\\n\\midrule\n" + rows +
            "\n\\botrule\n\\end{tabular}\n\\end{table}\n")


def read_jsonl(paths):
    rows = []
    for p in paths:
        with open(p) as f:
            rows += [json.loads(line) for line in f if line.strip()]
    return pd.DataFrame(rows)


def jload(path):
    with open(path) as f:
        return json.load(f)


def ca(df, r):
    return float(((df["correct"] == 1) & (df["radius"] >= r)).mean()) if len(df) else np.nan


def acr(df):
    return float((df["radius"] * df["correct"]).mean()) if len(df) else np.nan


def score_matrix(dm):
    """sigma x images matrix of 'radius if correct else -1' (images aligned by idx)."""
    piv = dm.assign(s=np.where(dm["correct"] == 1, dm["radius"], -1.0)).pivot_table(
        index="sigma", columns="idx", values="s", aggfunc="first")
    return piv.dropna(axis=1).values, list(piv.index)


def envelope(dm, radii):
    M, _ = score_matrix(dm)
    return np.array([(M >= r).mean(axis=1).max() for r in radii]) if M.size else np.full(len(radii), np.nan)


def envelope_ci(dm, radii, B=500, seed=0):
    M, _ = score_matrix(dm)
    if not M.size:
        return np.full(len(radii), np.nan), np.full(len(radii), np.nan)
    rng = np.random.default_rng(seed)
    boots = np.empty((B, len(radii)))
    for b in range(B):
        Mb = M[:, rng.integers(0, M.shape[1], M.shape[1])]
        boots[b] = [(Mb >= r).mean(axis=1).max() for r in radii]
    return np.percentile(boots, 2.5, axis=0), np.percentile(boots, 97.5, axis=0)


def boot_ci(x, B=1000, seed=0):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    m = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)


def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def methods_in(df):
    return [m for m in ORDER if m in set(df["method"])] + sorted(set(df["method"]) - set(ORDER))


def setting_files(res, prefix):
    """{(dataset, model): [files]} for cert_/ablation_ jsonl files."""
    out = {}
    for f in sorted(glob.glob(os.path.join(res, f"{prefix}_*_s*.jsonl"))):
        parts = os.path.basename(f)[len(prefix) + 1:].rsplit("_s", 1)[0].split("_")
        out.setdefault((parts[0], parts[1]), []).append(f)
    return out


def label(ds, model):
    return f"{'PLIP' if model == 'plip' else 'CLIP ViT-B/32'} / {'PathMNIST' if ds == 'pathmnist' else 'BloodMNIST'}"


# ----------------------------------------------------------------------------------------------- sections
def sec_data(R):
    for ds in ["pathmnist", "bloodmnist"]:
        p = os.path.join(R.res, f"datastats_{ds}.json")
        if not os.path.exists(p):
            continue
        st = jload(p)
        df = pd.DataFrame(dict(Class=st["classes"], Test=st["test_class_counts"],
                               Certified_subset=st["certified_subset_class_counts"],
                               Few_shot=[st["shots"]] * len(st["classes"])))
        t = R.savetab(df, f"data_{ds}", f"{ds}: class distribution (test split, certified subset, few-shot).",
                      f"tab:data_{ds}", 0)
        R.add("3.1 Data", f"{ds}: classes and splits (image size {st['img_size']})", table=t)
        for kind in ["samples", "noise_levels"]:
            f = os.path.join(R.res, f"{kind}_{ds}.png")
            if os.path.exists(f):
                R.add("3.1 Data", f"{ds}: {kind.replace('_', ' ')}", fig=os.path.relpath(f, R.dir))


def sec_sanity(R):
    p = os.path.join(R.res, "sanity.json")
    if os.path.exists(p):
        s = jload(p)
        df = pd.DataFrame([dict(Check="min cosine (learnable vs hand-crafted prompt)",
                                Value=s["min_cosine_learnable_vs_reference"]),
                           dict(Check="zero-shot clean accuracy (%) on certified subset",
                                Value=100 * s["zeroshot_clean_acc"])] +
                          [dict(Check=f"max certifiable radius, sigma={k}", Value=v)
                           for k, v in s["max_radius_at_sigma"].items()])
        R.add("4.3 Sanity", "Sanity checks", table=R.savetab(df, "sanity", "Sanity checks.", "tab:sanity", 3))


def sec_reproduction(R, df):
    have = [m for m in ["zs_plip", "coop", "ps_zeroshot", "ps_fewshot", "ps_combined"] if m in set(df["method"])]
    if not have:
        return
    radii = REPORTED["radii"]
    rows, fig, axes = [], *plt.subplots(1, len(have), figsize=(3.0 * len(have), 2.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, have):
        ours = 100 * envelope(df[df.method == m], radii)
        lo, hi = envelope_ci(df[df.method == m], radii)
        ax.plot(radii, REPORTED[m], "o--", color="k", label="reported", ms=4)
        ax.plot(radii, ours, "s-", color=COLORS[m], label="ours", ms=4)
        ax.fill_between(radii, 100 * lo, 100 * hi, color=COLORS[m], alpha=0.2)
        ax.set_title(NAMES[m], fontsize=9); ax.set_xlabel("L2 radius"); ax.set_ylim(0, 100)
        for r, a, b in zip(radii, REPORTED[m], ours):
            rows.append(dict(Method=NAMES[m], Radius=r, Reported=a, Reproduced=b, Difference=b - a))
    axes[0].set_ylabel("Certified accuracy (%)"); axes[0].legend(frameon=False)
    f = R.savefig(fig, "reproduction_curves")
    long = pd.DataFrame(rows)
    wide = long.pivot_table(index="Method", columns="Radius", values=["Reported", "Reproduced"], sort=False)
    wide.columns = [f"{a[:3]}@{b}" for a, b in wide.columns]
    wide = wide.reset_index()
    t = R.savetab(wide, "reproduction", "PromptSmooth reproduction: reported vs reproduced certified accuracy (%), "
                  "envelope over sigma.", "tab:repro")
    mad = long.groupby("Method")["Difference"].apply(lambda d: d.abs().mean()).reset_index(name="Mean |diff| (pp)")
    t2 = R.savetab(mad, "reproduction_error", "Mean absolute reproduction error (percentage points).",
                   "tab:repro_err")
    R.add("4.3 Baseline reproduction", "Reported vs reproduced (shaded: 95% bootstrap CI)", fig=f, table=t)
    R.add("4.3 Baseline reproduction", "Mean absolute reproduction error", table=t2)


def sec_main(R, df, ds, model, tagname, section=None):
    meths = methods_in(df)
    sigmas = sorted(df["sigma"].unique())
    fine = np.linspace(0, 2.0, 101)
    sec = section or f"4.4 Comparison — {label(ds, model)}"
    # envelope curves
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    for m in meths:
        ax.plot(fine, 100 * envelope(df[df.method == m], fine), STYLE.get(m, "-"), color=COLORS.get(m),
                label=NAMES.get(m, m), lw=1.8)
    if ds == "pathmnist" and model == "plip":
        for m in ["denoised_smoothing", "diffusion_smoothing"]:
            ax.plot(REPORTED["radii"], REPORTED[m], "x:", color=COLORS[m], label=NAMES[m], ms=4, lw=0.8)
    ax.set_xlabel("L2 radius"); ax.set_ylabel("Certified accuracy (%)"); ax.set_ylim(0, 100)
    ax.legend(fontsize=7, frameon=False, ncol=2); ax.grid(alpha=0.25)
    ax.set_title(f"Certified accuracy, envelope over σ — {label(ds, model)}", fontsize=9)
    f1 = R.savefig(fig, f"envelope_{tagname}")
    # table with CIs
    rows = []
    for m in meths:
        d = df[df.method == m]
        env = 100 * envelope(d, RADII)
        lo, hi = envelope_ci(d, [0.5])
        best_acr = max(acr(d[d.sigma == s]) for s in sigmas)
        row = dict(Method=NAMES.get(m, m), **{f"r={r}": v for r, v in zip(RADII, env)})
        row["ACR (best σ)"] = best_acr
        row["95% CI @0.5"] = f"[{100 * lo[0]:.1f}, {100 * hi[0]:.1f}]"
        rows.append(row)
    t1 = R.savetab(pd.DataFrame(rows), f"main_{tagname}", f"Certified accuracy (%) and ACR, {label(ds, model)}.",
                   f"tab:main_{tagname}")
    R.add(sec, "Certified accuracy vs radius (envelope over σ)", fig=f1, table=t1,
          note="† TTA variants: certificate for the adapted prompt held fixed (Remark 1).")
    # per-sigma panels
    fig, axes = plt.subplots(1, len(sigmas), figsize=(3.0 * len(sigmas), 2.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, s in zip(axes, sigmas):
        for m in meths:
            d = df[(df.method == m) & (df.sigma == s)]
            ax.plot(fine, [100 * ca(d, r) for r in fine], STYLE.get(m, "-"), color=COLORS.get(m), lw=1.5,
                    label=NAMES.get(m, m))
        ax.set_title(f"σ = {s}"); ax.set_xlabel("L2 radius"); ax.set_ylim(0, 100); ax.grid(alpha=0.25)
    axes[0].set_ylabel("Certified accuracy (%)"); axes[-1].legend(fontsize=6, frameon=False)
    f2 = R.savefig(fig, f"per_sigma_{tagname}")
    rows = []
    for m in meths:
        for s in sigmas:
            d = df[(df.method == m) & (df.sigma == s)]
            rows.append(dict(Method=NAMES.get(m, m), sigma=s, **{"CA(0)": 100 * ca(d, 0.0)},
                             Abstain=100 * float((d["predict"] == -1).mean()), ACR=acr(d),
                             **{f"CA({r})": 100 * ca(d, r) for r in [0.25, 0.5, 1.0]}))
    t2 = R.savetab(pd.DataFrame(rows), f"per_sigma_{tagname}", f"Per-sigma results, {label(ds, model)}.",
                   f"tab:persigma_{tagname}", 2)
    R.add(sec, "Per-σ certified accuracy, abstention and ACR", fig=f2, table=t2)
    # ACR bars with bootstrap CIs
    fig, ax = plt.subplots(figsize=(6.5, 3.0))
    w = 0.8 / len(meths)
    for j, m in enumerate(meths):
        vals, errs = [], [[], []]
        for s in sigmas:
            d = df[(df.method == m) & (df.sigma == s)]
            v = (d["radius"] * d["correct"]).values
            a = v.mean() if len(v) else np.nan
            lo, hi = boot_ci(v) if len(v) else (np.nan, np.nan)
            vals.append(a); errs[0].append(a - lo); errs[1].append(hi - a)
        ax.bar(np.arange(len(sigmas)) + j * w, vals, w, yerr=errs, color=COLORS.get(m), label=NAMES.get(m, m),
               capsize=2, edgecolor="white", lw=0.5)
    ax.set_xticks(np.arange(len(sigmas)) + w * (len(meths) - 1) / 2); ax.set_xticklabels([f"σ={s}" for s in sigmas])
    ax.set_ylabel("ACR"); ax.legend(fontsize=6, frameon=False, ncol=2)
    f3 = R.savefig(fig, f"acr_bars_{tagname}")
    R.add(sec, "Average certified radius (95% bootstrap CI)", fig=f3)


def sec_paired(R, df, tagname, a="coreprompt", b="ps_fewshot"):
    if not {a, b} <= set(df["method"]):
        return
    rows = []
    sigmas = sorted(df["sigma"].unique())
    fig, axes = plt.subplots(1, len(sigmas), figsize=(2.8 * len(sigmas), 2.8))
    axes = np.atleast_1d(axes)
    for ax, s in zip(axes, sigmas):
        da = df[(df.method == a) & (df.sigma == s)].set_index("idx")
        db = df[(df.method == b) & (df.sigma == s)].set_index("idx")
        common = da.index.intersection(db.index)
        ra = np.where(da.loc[common, "correct"] == 1, da.loc[common, "radius"], 0.0)
        rb = np.where(db.loc[common, "correct"] == 1, db.loc[common, "radius"], 0.0)
        for r in [0.25, 0.5]:
            ia, ib = ra >= max(r, 1e-9), rb >= max(r, 1e-9)
            n10, n01 = int((ia & ~ib).sum()), int((~ia & ib).sum())
            p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
            lo, hi = boot_ci(ra - rb)
            rows.append(dict(sigma=s, radius=r, **{"CoRe only": n10, "PS only": n01},
                             **{"McNemar p": p, "ΔACR": float((ra - rb).mean()), "ΔACR 95% CI": f"[{lo:.3f}, {hi:.3f}]"}))
        ax.scatter(rb, ra, s=6, alpha=0.5, color=COLORS[a])
        mx = max(ra.max(initial=0), rb.max(initial=0), 0.1)
        ax.plot([0, mx], [0, mx], "k--", lw=0.8)
        ax.set_title(f"σ = {s}"); ax.set_xlabel("radius, PS few-shot")
    axes[0].set_ylabel("radius, CoRe-Prompt")
    f = R.savefig(fig, f"paired_{tagname}")
    t = R.savetab(pd.DataFrame(rows), f"paired_{tagname}", "Paired comparison of CoRe-Prompt and PromptSmooth "
                  "few-shot on the same images and noise (McNemar exact test; paired bootstrap CI of the ACR "
                  "difference).", f"tab:paired_{tagname}", 3)
    R.add("4.4 Comparison — paired test", "Per-image certified radius: CoRe-Prompt vs PS few-shot "
          "(points above the diagonal favour CoRe-Prompt)", fig=f, table=t)


def sec_fulltest(R, path, ds, model, tagname, classnames=None, section=None):
    sec = section or f"4.4 Full test split — {label(ds, model)}"
    z = np.load(path)
    y = z["labels"]
    keys = [k for k in z.files if k.startswith("votes|")]
    meths = [m for m in ORDER if any(k.split("|")[1] == m for k in keys)]
    sigmas = sorted({float(k.split("|")[2]) for k in keys})
    rows = []
    fig, ax = plt.subplots(figsize=(5.2, 3.3))
    for m in meths:
        accs, los, his = [], [], []
        for s in sigmas:
            v = z[f"votes|{m}|{s}"]
            nv = v.sum(1).max()
            vote = (v.argmax(1) == y)
            single = v[np.arange(len(y)), y].sum() / (nv * len(y))
            clean = (z[f"clean|{m}|{s}"] == y).mean()
            lo, hi = wilson(vote.sum(), len(y))
            accs.append(100 * vote.mean()); los.append(100 * lo); his.append(100 * hi)
            rows.append(dict(Method=NAMES.get(m, m), sigma=s, Clean=100 * clean, **{"Noisy (1 draw)": 100 * single},
                             **{"Noisy vote": 100 * vote.mean(), "95% CI": f"[{100 * lo:.1f}, {100 * hi:.1f}]"}))
        ax.plot(sigmas, accs, "o" + STYLE.get(m, "-"), color=COLORS.get(m), label=NAMES.get(m, m), ms=3)
        ax.fill_between(sigmas, los, his, color=COLORS.get(m), alpha=0.15)
    ax.set_xlabel("σ"); ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 100); ax.grid(alpha=0.25)
    ax.legend(fontsize=7, frameon=False, loc="lower left")
    ax.set_title(f"Complete test split (N = {len(y)}), noisy-vote accuracy — {label(ds, model)}", fontsize=8)
    f = R.savefig(fig, f"fulltest_{tagname}")
    t = R.savetab(pd.DataFrame(rows), f"fulltest_{tagname}", f"Accuracy (%) on the complete test split "
                  f"(N = {len(y)}), {label(ds, model)}: clean base classifier, single noisy draw, and majority "
                  "vote over noisy draws (Wilson 95% CI).", f"tab:full_{tagname}")
    R.add(sec, f"Accuracy on all {len(y)} test images", fig=f, table=t)
    # per-class heatmap + confusion matrices at the middle sigma
    s = sigmas[len(sigmas) // 2]
    C = int(max(y.max() + 1, z[f"votes|{meths[0]}|{s}"].shape[1]))
    names = classnames or [str(c) for c in range(C)]
    H = np.array([[100 * (z[f"votes|{m}|{s}"].argmax(1)[y == c] == c).mean() if (y == c).any() else np.nan
                   for c in range(C)] for m in meths])
    fig, ax = plt.subplots(figsize=(1.0 + 0.65 * C, 0.6 + 0.4 * len(meths)))
    im = ax.imshow(H, vmin=0, vmax=100, cmap="viridis", aspect="auto")
    ax.set_xticks(range(C)); ax.set_xticklabels([n[:14] for n in names], rotation=40, ha="right", fontsize=7)
    ax.set_yticks(range(len(meths))); ax.set_yticklabels([NAMES.get(m, m) for m in meths], fontsize=7)
    for i in range(len(meths)):
        for j in range(C):
            if not np.isnan(H[i, j]):
                ax.text(j, i, f"{H[i, j]:.0f}", ha="center", va="center", fontsize=6,
                        color="white" if H[i, j] < 60 else "black")
    fig.colorbar(im, ax=ax, fraction=0.03); ax.set_title(f"Per-class noisy-vote accuracy (%), σ = {s}", fontsize=8)
    f2 = R.savefig(fig, f"perclass_{tagname}")
    R.add(sec, f"Per-class accuracy at σ = {s}", fig=f2)
    pair = [m for m in ["ps_fewshot", "coreprompt"] if m in meths]
    if pair:
        fig, axes = plt.subplots(1, len(pair), figsize=(3.6 * len(pair), 3.3))
        axes = np.atleast_1d(axes)
        for ax, m in zip(axes, pair):
            pred = z[f"votes|{m}|{s}"].argmax(1)
            cm = np.zeros((C, C))
            np.add.at(cm, (y, pred), 1)
            cm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
            ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
            ax.set_xticks(range(C)); ax.set_yticks(range(C))
            ax.set_xticklabels([n[:4] for n in names], fontsize=6, rotation=90); ax.set_yticklabels([n[:10] for n in names], fontsize=6)
            ax.set_xlabel("predicted"); ax.set_title(NAMES.get(m, m), fontsize=8)
        axes[0].set_ylabel("true")
        f3 = R.savefig(fig, f"confusion_{tagname}")
        R.add(sec, f"Row-normalised confusion matrices at σ = {s}", fig=f3)


def sec_attacks(R, path, cert):
    res = jload(path)
    tr = {k: v for k, v in res.items() if k.startswith("transform|")}
    if tr:
        rows = []
        for k, v in tr.items():
            _, d, mode, e = k.split("|")
            rows.append(dict(defence=d, mode=mode, eps=float(e.split("=")[1]) * 255, acc=100 * v))
        d = pd.DataFrame(rows)
        defs = [x for x in ["none", "bit_depth", "jpeg", "resize_pad"] if x in set(d.defence)]
        fig, axes = plt.subplots(1, len(defs), figsize=(2.8 * len(defs), 2.8), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, df_ in zip(axes, defs):
            g = d[d.defence == df_]
            clean = g[g["mode"] == "clean"]["acc"].mean()
            for mode, st, c in [("nonadaptive", "s--", "#E69F00"), ("adaptive", "o-", "#D55E00")]:
                gm = g[g["mode"] == mode].sort_values("eps")
                ax.plot([0] + gm["eps"].tolist(), [clean] + gm["acc"].tolist(), st, color=c, ms=3,
                        label="non-adaptive (transfer)" if mode == "nonadaptive" else "adaptive (BPDA/EOT)")
            ax.set_title(df_.replace("_", " ")); ax.set_xlabel("ε (×1/255, L∞)"); ax.set_ylim(0, 100)
            ax.grid(alpha=0.25)
        axes[0].set_ylabel("Robust accuracy (%)"); axes[0].legend(fontsize=7, frameon=False)
        f = R.savefig(fig, "attacks_transform")
        wide = d.pivot_table(index="defence", columns=["mode", "eps"], values="acc")
        wide.columns = [f"{m[:2].upper()}@{e:.0f}" if m != "clean" else "clean" for m, e in wide.columns]
        wide = wide.loc[[x for x in defs]].reset_index()
        t = R.savetab(wide, "attacks_transform", "Robust accuracy (%) of zero-shot PLIP with input-transformation "
                      "defences; NO = non-adaptive, AD = adaptive (BPDA/EOT); ε in units of 1/255.",
                      "tab:transform")
        R.add("4.5 Robustness — adaptive vs non-adaptive", "Input-transformation defences", fig=f, table=t)
        gap = d[d["mode"] != "clean"].pivot_table(index=["defence", "eps"], columns="mode", values="acc").reset_index()
        gap["gap"] = gap["nonadaptive"] - gap["adaptive"]
        fig, ax = plt.subplots(figsize=(5.5, 2.8))
        epss = sorted(gap["eps"].unique())
        w = 0.8 / max(len(defs), 1)
        for j, df_ in enumerate(defs):
            g = gap[gap.defence == df_].set_index("eps").reindex(epss)
            ax.bar(np.arange(len(epss)) + j * w, g["gap"], w, label=df_.replace("_", " "))
        ax.set_xticks(np.arange(len(epss)) + w * (len(defs) - 1) / 2)
        ax.set_xticklabels([f"{e:.0f}/255" for e in epss]); ax.set_ylabel("Adaptive gap (pp)")
        ax.axhline(0, color="k", lw=0.6); ax.legend(fontsize=7, frameon=False)
        f2 = R.savefig(fig, "attacks_gap")
        R.add("4.5 Robustness — adaptive vs non-adaptive", "Adaptive gap = non-adaptive − adaptive accuracy "
              "(over-estimation by non-adaptive evaluation)", fig=f2)
    sm = {k: v for k, v in res.items() if k.startswith("smoothed|")}
    if sm:
        rows = []
        for k, v in sm.items():
            _, m, s, mode, e = k.split("|")
            rows.append(dict(method=m, sigma=float(s.split("=")[1]), mode=mode, eps=float(e.split("=")[1]), acc=100 * v))
        d = pd.DataFrame(rows)
        sigmas = sorted(d.sigma.unique())
        meths = [m for m in ORDER if m in set(d.method)]
        fig, axes = plt.subplots(1, len(sigmas), figsize=(3.6 * len(sigmas), 3.0), sharey=True)
        axes = np.atleast_1d(axes)
        trows = []
        for ax, s in zip(axes, sigmas):
            for m in meths:
                g = d[(d.method == m) & (d.sigma == s)]
                clean = g[g["mode"] == "clean"]["acc"].mean()
                epss = sorted(g[g["mode"] != "clean"]["eps"].unique())
                na = [clean] + [g[(g["mode"] == "nonadaptive") & (g.eps == e)]["acc"].mean() for e in epss]
                ad = [clean] + [g[(g["mode"] == "adaptive") & (g.eps == e)]["acc"].mean() for e in epss]
                xs = [0] + epss
                ax.plot(xs, na, "s--", color=COLORS.get(m), ms=3, lw=1)
                ax.plot(xs, ad, "o-", color=COLORS.get(m), ms=3, lw=1.6, label=NAMES.get(m, m))
                cc = cert[(cert.method == m) & (cert.sigma == s)] if cert is not None else pd.DataFrame()
                if len(cc):
                    ax.plot(xs, [100 * ca(cc, e) for e in xs], ":", color=COLORS.get(m), lw=1.2)
                for e, a1, a2 in zip(xs[1:], na[1:], ad[1:]):
                    cv = 100 * ca(cc, e) if len(cc) else np.nan
                    trows.append(dict(Method=NAMES.get(m, m), sigma=s, eps=e, **{"Non-adaptive": a1, "SmoothAdv": a2,
                                      "Certified (lower bound)": cv,
                                      "check AD ≥ CA−5": "ok" if np.isnan(cv) or a2 >= cv - 5 else "VIOLATED"}))
            ax.set_title(f"σ = {s}  (dashed: non-adaptive, solid: SmoothAdv, dotted: certified)", fontsize=7)
            ax.set_xlabel("ε (L2)"); ax.set_ylim(0, 100); ax.grid(alpha=0.25)
        axes[0].set_ylabel("Accuracy of g (%)"); axes[0].legend(fontsize=7, frameon=False)
        f = R.savefig(fig, "attacks_smoothed")
        t = R.savetab(pd.DataFrame(trows), "attacks_smoothed", "Empirical robust accuracy (%) of smoothed "
                      "classifiers (PREDICT, abstention = error) vs certified accuracy at the same radius; the check "
                      "column verifies that the attack does not beat the certificate (5 pp Monte-Carlo slack).",
                      "tab:smoothattack")
        R.add("4.5 Robustness — smoothed classifiers", "Non-adaptive vs adaptive (SmoothAdv) vs certified lower bound",
              fig=f, table=t)


def sec_ablation(R, df):
    abl = df[df.method.isin(ABL)]
    if len(abl):
        rows = [dict(Objective=ABL[m], **{f"r={r}": 100 * ca(abl[abl.method == m], r) for r in [0, 0.25, 0.5, 1.0]},
                     ACR=acr(abl[abl.method == m])) for m in ABL if m in set(abl.method)]
        t = pd.DataFrame(rows)
        fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), gridspec_kw=dict(width_ratios=[3, 1]))
        rr = [0, 0.25, 0.5, 1.0]
        w = 0.8 / len(rows)
        for j, r in enumerate(rows):
            axes[0].bar(np.arange(len(rr)) + j * w, [r[f"r={x}"] for x in rr], w, label=r["Objective"])
            axes[1].bar(j, r["ACR"])
        axes[0].set_xticks(np.arange(len(rr)) + w * (len(rows) - 1) / 2)
        axes[0].set_xticklabels([f"r={x}" for x in rr]); axes[0].set_ylabel("Certified accuracy (%)")
        axes[0].legend(fontsize=7, frameon=False); axes[1].set_xticks(range(len(rows)))
        axes[1].set_xticklabels(["CE", "+con", "+rad", "full"][:len(rows)]); axes[1].set_ylabel("ACR")
        f = R.savefig(fig, "ablation")
        R.add("4.5 Ablation", "Loss-term ablation at σ = 0.5",
              fig=f, table=R.savetab(t, "ablation", "Loss ablation at sigma = 0.5.", "tab:ablation"))
    sens = df[df.method.str.startswith("sens|")]
    if len(sens):
        full = df[df.method == "abl_full"]
        rows = []
        for m, g in sens.groupby("method"):
            _, key, val = m.split("|")
            rows.append(dict(parameter=key, value=float(val), ACR=acr(g), **{"CA(0.25)": 100 * ca(g, 0.25),
                             "CA(0.5)": 100 * ca(g, 0.5), "CA(1.0)": 100 * ca(g, 1.0)}))
        for key, dv in DEFAULTS.items():
            if dv is not None and len(full) and key in {r["parameter"] for r in rows}:
                rows.append(dict(parameter=key, value=float(dv), ACR=acr(full), **{"CA(0.25)": 100 * ca(full, 0.25),
                                 "CA(0.5)": 100 * ca(full, 0.5), "CA(1.0)": 100 * ca(full, 1.0)}, default=True))
        t = pd.DataFrame(rows).sort_values(["parameter", "value"])
        t["default"] = t.get("default", pd.Series(dtype=object)).fillna(False)
        keys = list(dict.fromkeys(t.parameter))
        fig, axes = plt.subplots(1, len(keys), figsize=(2.3 * len(keys), 2.5), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, key in zip(axes, keys):
            g = t[t.parameter == key]
            ax.plot(g["value"], g["CA(0.5)"], "o-", color=COLORS["coreprompt"], ms=3)
            dg = g[g["default"] == True]  # noqa: E712
            ax.plot(dg["value"], dg["CA(0.5)"], "*", color="#D55E00", ms=10)
            ax.set_title(key, fontsize=8); ax.grid(alpha=0.25)
            if key in ("lam_con", "eta_ent", "mu_rad"):
                ax.set_xscale("symlog", linthresh=0.5)
        axes[0].set_ylabel("CA at r = 0.5 (%)")
        f = R.savefig(fig, "sensitivity")
        R.add("4.5 Sensitivity", "One-at-a-time sensitivity at σ = 0.5 (star = default)", fig=f,
              table=R.savetab(t.drop(columns="default"), "sensitivity", "Hyper-parameter sensitivity at sigma = 0.5.",
                              "tab:sens", 2))


def sec_generalisation(R, settings):
    rows = []
    gens = [(k, v) for k, v in settings.items() if k != ("pathmnist", "plip")]
    if not gens:
        return
    fig, axes = plt.subplots(1, len(gens), figsize=(4.2 * len(gens), 3.0), sharey=True)
    axes = np.atleast_1d(axes)
    fine = np.linspace(0, 2.0, 81)
    for ax, ((ds, model), df) in zip(axes, gens):
        for m in methods_in(df):
            d = df[df.method == m]
            ax.plot(fine, 100 * envelope(d, fine), STYLE.get(m, "-"), color=COLORS.get(m), label=NAMES.get(m, m))
            env = 100 * envelope(d, [0, 0.25, 0.5, 1.0])
            rows.append(dict(Setting=label(ds, model), Method=NAMES.get(m, m),
                             **{f"r={r}": v for r, v in zip([0, 0.25, 0.5, 1.0], env)},
                             ACR=max(acr(d[d.sigma == s]) for s in d.sigma.unique())))
        ax.set_title(label(ds, model), fontsize=9); ax.set_xlabel("L2 radius"); ax.set_ylim(0, 100)
        ax.grid(alpha=0.25)
    axes[0].set_ylabel("Certified accuracy (%)"); axes[-1].legend(fontsize=6, frameon=False)
    f = R.savefig(fig, "generalisation")
    R.add("4.5 Generalisation", "Cross-model and cross-dataset certified accuracy", fig=f,
          table=R.savetab(pd.DataFrame(rows), "generalisation", "Generalisation: certified accuracy (%) and ACR.",
                          "tab:general"))


def sec_training(R):
    for p in sorted(glob.glob(os.path.join(R.res, "trainlogs_*.json"))):
        tag = os.path.basename(p)[10:-5]
        logs = jload(p)
        sigmas = sorted(logs, key=float)
        fig, axes = plt.subplots(2, len(sigmas), figsize=(2.8 * len(sigmas), 4.2), squeeze=False)
        for j, s in enumerate(sigmas):
            for m, lg in logs[s].items():
                if not lg.get("epoch_loss"):
                    continue
                axes[0, j].plot(lg["epoch_loss"], color=COLORS.get(m, None), label=NAMES.get(m, m))
                if any(v is not None for v in lg.get("soft_radius", [])):
                    axes[1, j].plot([np.nan if v is None else v for v in lg["soft_radius"]], color=COLORS.get(m))
            axes[0, j].set_title(f"σ = {s}"); axes[1, j].set_xlabel("epoch")
        axes[0, 0].set_ylabel("training loss"); axes[1, 0].set_ylabel("soft certified radius\n(CoRe-Prompt)")
        axes[0, -1].legend(fontsize=6, frameon=False)
        R.add("4.5 Training dynamics", f"Training curves ({tag})", fig=R.savefig(fig, f"training_{tag}"))


def sec_efficiency(R):
    p = os.path.join(R.res, "efficiency.json")
    if os.path.exists(p):
        e = jload(p)
        t = pd.DataFrame(e["rows"])
        t["method"] = t["method"].map(lambda m: NAMES.get(m, m))
        extra = pd.DataFrame([dict(method="Certification, one method (s/img)", train_time_s=e["cert_s_per_image_one_method"]),
                              dict(method=f"Certification, {e['n_fixed_methods']} methods shared (s/img)",
                                   train_time_s=e["cert_s_per_image_all_fixed_shared"]),
                              dict(method="Test-time adaptation (s/img)", train_time_s=e["tta_s_per_image"])])
        fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
        axes[0].barh(t["method"], t["train_time_s"], color="#0072B2"); axes[0].set_xlabel("training time (s)")
        k = e["n_fixed_methods"]
        axes[1].bar(["1 method", f"{k} methods\nseparately", f"{k} methods\nshared"],
                    [e["cert_s_per_image_one_method"], k * e["cert_s_per_image_one_method"],
                     e["cert_s_per_image_all_fixed_shared"]], color=["#999999", "#E69F00", "#009E73"])
        axes[1].set_ylabel(f"certification s / image (n = {e['n']})")
        f = R.savefig(fig, "efficiency")
        R.add("4.5 Efficiency", f"Computational cost ({e['gpu']})", fig=f,
              table=R.savetab(pd.concat([t, extra], ignore_index=True), "efficiency", "Computational cost.",
                              "tab:eff", 2))
    p = os.path.join(R.res, "timings.json")
    if os.path.exists(p):
        tm = jload(p)
        t = pd.DataFrame(dict(stage=list(tm), minutes=list(tm.values())))
        t = pd.concat([t, pd.DataFrame([dict(stage="total", minutes=t.minutes.sum())])], ignore_index=True)
        R.add("4.5 Efficiency", "Wall-clock time per stage (Colab)", table=R.savetab(t, "timings", "Stage times.",
                                                                                     "tab:timings", 1))


# ----------------------------------------------------------------------------------------------- driver
def make_report(out_dir):
    R = Report(out_dir)
    classnames = {}
    for ds in ["pathmnist", "bloodmnist"]:
        p = os.path.join(R.res, f"datastats_{ds}.json")
        if os.path.exists(p):
            classnames[ds] = jload(p)["classes"]
    cert = {k: read_jsonl(v) for k, v in setting_files(R.res, "cert").items()}
    steps = [("data", lambda: sec_data(R)), ("sanity", lambda: sec_sanity(R))]
    main = cert.get(("pathmnist", "plip"))
    if main is not None:
        steps += [("reproduction", lambda: sec_reproduction(R, main)),
                  ("main", lambda: sec_main(R, main, "pathmnist", "plip", "pathmnist_plip")),
                  ("paired", lambda: sec_paired(R, main, "pathmnist_plip"))]
    fulls = sorted(glob.glob(os.path.join(R.res, "fulltest_*.npz")), key=lambda p: "pathmnist_plip" not in p)
    main_full = [p for p in fulls if "pathmnist_plip" in p]
    for p in main_full:
        steps.append(("fulltest main", lambda p=p: sec_fulltest(R, p, "pathmnist", "plip", "pathmnist_plip",
                                                                 classnames.get("pathmnist"))))
    ap = os.path.join(R.res, "attacks_pathmnist_plip.json")
    if os.path.exists(ap):
        steps.append(("attacks", lambda: sec_attacks(R, ap, main)))
    abl = setting_files(R.res, "ablation").get(("pathmnist", "plip"))
    if abl:
        steps.append(("ablation", lambda: sec_ablation(R, read_jsonl(abl))))
    steps.append(("generalisation", lambda: sec_generalisation(R, cert)))
    for (ds, model), df in cert.items():
        if (ds, model) != ("pathmnist", "plip"):
            steps.append((f"gen {ds}/{model}", lambda df=df, ds=ds, model=model: sec_main(
                R, df, ds, model, f"{ds}_{model}", section=f"4.5 Generalisation — {label(ds, model)}")))
    for p in [q for q in fulls if q not in main_full]:
        ds, model = os.path.basename(p)[9:-4].split("_")
        steps.append((f"fulltest {ds}/{model}",
                      lambda p=p, ds=ds, model=model: sec_fulltest(R, p, ds, model, f"{ds}_{model}", classnames.get(ds),
                                                                   section=f"4.5 Generalisation — {label(ds, model)}")))
    steps += [("training", lambda: sec_training(R)), ("efficiency", lambda: sec_efficiency(R))]
    for name, fn in steps:
        try:
            fn()
        except Exception:
            print(f"[report] section '{name}' skipped:\n{traceback.format_exc(limit=2)}")
    with open(os.path.join(R.dir, "index.json"), "w") as f:
        json.dump(R.index, f, indent=1)
    print(f"[report] {len(R.index)} items -> {R.dir}")
    return R.index


def show(out_dir):
    """Display the report inline in a notebook (figures + tables in paper order)."""
    from IPython.display import Image, Markdown, display
    rdir = os.path.join(out_dir, "report")
    index = jload(os.path.join(rdir, "index.json"))
    sec = None
    for it in index:
        if it["section"] != sec:
            sec = it["section"]
            display(Markdown(f"## {sec}"))
        display(Markdown(f"**{it['title']}**" + (f"  \n_{it['note']}_" if it.get("note") else "")))
        if it.get("figure"):
            display(Image(filename=os.path.join(rdir, it["figure"])))
        if it.get("table"):
            df = pd.read_csv(os.path.join(rdir, it["table"]))
            display(df.style.format(precision=2, na_rep="–").hide(axis="index"))

In [ ]:
%%writefile coreprompt/run.py
"""Experiment stages (run in this order by notebooks/CoRe-Prompt_colab.ipynb).

    python -m coreprompt.run <stage> --out_dir DIR [--preset colab|smoke] [--data_dir DIR] [--set key=value ...]

  stage       paper   what it does (light-weight, free-tier version)
  sanity      4.3     prompt-encoder equivalence check, zero-shot accuracy, max certifiable radii
  datastats   3.1     dataset statistics, sample / noise-level figures (PathMNIST and BloodMNIST)
  certify     4.3-4.4 [1] PromptSmooth reproduction + [2] CoRe-Prompt; 500 images, 4 sigmas, n = 2,000
  fulltest    4.4     every fixed-prompt method on the complete test split: clean, noisy and noisy-vote accuracy
  attacks     4.5     [3] adaptive vs non-adaptive attacks (transform defences, smoothed classifiers); 100 images
  ablation    4.5     [4] loss ablation + sensitivity at sigma = 0.5; 200 images
  gen_clip    4.5     [5] cross-model generalisation (CLIP ViT-B/32 on PathMNIST)
  gen_blood   4.5     [5] cross-dataset generalisation (PLIP on BloodMNIST)
  efficiency  4.5     parameters, training / certification / TTA time, memory
  report      4       all tables (CSV + LaTeX) and figures

Every stage is checkpointed under OUT_DIR and resumes when re-run.
"""
import argparse
import ast
import dataclasses
import json
import os
import time

import numpy as np
import torch

from . import attacks as A
from .config import STAGES, make_config
from .data import iterate_full, load_data
from .methods import ALL_METHODS, build
from .models import MedVLM, check_equivalence
from .smoothing import certify_methods, max_certifiable_radius
from .train import text_feats_for, train_prompt, zeroshot_text_feats
from .utils import get_device, load_json, results_path, save_json, set_seed

FIXED_METHODS = ["zs_plip", "coop", "ps_fewshot", "coreprompt"]          # prompts that do not depend on the input
GEN_METHODS = ["zs_plip", "coop", "ps_fewshot", "ps_combined", "coreprompt", "coreprompt_tta"]


def _tag(cfg):
    return f"{cfg.dataset}_{cfg.model}"


def _setup(cfg):
    set_seed(cfg.seed)
    vlm = MedVLM(cfg.model, get_device())
    return vlm, load_data(cfg)


# ------------------------------------------------------------------------------------------------ sanity
def exp_sanity(cfg):
    vlm, data = _setup(cfg)
    cos = check_equivalence(vlm, data["classnames"], data["noisy_init"])
    t = zeroshot_text_feats(vlm, data)
    correct = 0
    with torch.no_grad():
        for i in range(0, len(data["y_test"]), 100):
            x = data["x_test"][i:i + 100].to(vlm.device)
            correct += (vlm.encode_image(x) @ t.t()).argmax(-1).cpu().eq(data["y_test"][i:i + 100]).sum().item()
    res = dict(min_cosine_learnable_vs_reference=cos, zeroshot_clean_acc=correct / len(data["y_test"]),
               n_images=len(data["y_test"]), img_size=data["stats"]["img_size"],
               max_radius_at_sigma={str(s): max_certifiable_radius(cfg.n, cfg.alpha, s) for s in cfg.sigmas})
    save_json(res, results_path(cfg, "sanity.json"))
    assert cos > 0.999, f"prompt learner does not reproduce the hand-crafted prompt (cos={cos})"
    print(json.dumps(res, indent=1))


# ------------------------------------------------------------------------------------------------ data
def exp_datastats(cfg):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for ds in ["pathmnist", "bloodmnist"]:
        c2 = dataclasses.replace(cfg, dataset=ds, n_full=0)
        data = load_data(c2)
        names = data["classnames"]
        st = dict(dataset=ds, classes=names, img_size=data["stats"]["img_size"], shots=cfg.shots,
                  n_shot_images=len(data["y_shot"]), test_split_total=data["stats"]["test_total"],
                  test_class_counts=data["stats"]["test_class_counts"],
                  certified_subset_class_counts=np.bincount(data["y_test"].numpy(), minlength=len(names)).tolist())
        save_json(st, results_path(cfg, f"datastats_{ds}.json"))
        C = len(names)
        fig, ax = plt.subplots(1, C, figsize=(1.5 * C, 1.9))
        for c in range(C):
            i = int((data["y_shot"] == c).nonzero()[0, 0])
            ax[c].imshow(data["x_shot"][i].permute(1, 2, 0).numpy())
            ax[c].set_title(names[c][:16], fontsize=7)
            ax[c].axis("off")
        fig.tight_layout()
        fig.savefig(results_path(cfg, f"samples_{ds}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)
        x = data["x_shot"][0]
        sig = [0.1, 0.25, 0.5, 1.0]
        fig, ax = plt.subplots(1, len(sig) + 1, figsize=(2 * (len(sig) + 1), 2.1))
        ax[0].imshow(x.permute(1, 2, 0).numpy()); ax[0].set_title("clean", fontsize=8); ax[0].axis("off")
        g = torch.Generator().manual_seed(0)
        for j, s in enumerate(sig):
            xn = (x + s * torch.randn(x.shape, generator=g)).clamp(0, 1)
            ax[j + 1].imshow(xn.permute(1, 2, 0).numpy()); ax[j + 1].set_title(f"σ = {s}", fontsize=8)
            ax[j + 1].axis("off")
        fig.tight_layout()
        fig.savefig(results_path(cfg, f"noise_levels_{ds}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(json.dumps(st))


# ------------------------------------------------------------------------------------------------ [1]+[2]
def exp_certify(cfg, methods=None):
    methods = methods or ALL_METHODS
    vlm, data = _setup(cfg)
    lp = results_path(cfg, f"trainlogs_{_tag(cfg)}.json")
    logs_all = load_json(lp) if os.path.exists(lp) else {}
    for s in cfg.sigmas:
        logs = {}
        m = build(vlm, data, cfg, s, methods, logs=logs)
        logs_all[str(s)] = {**logs_all.get(str(s), {}), **logs}
        save_json(logs_all, lp)
        certify_methods(vlm, data["x_test"], data["y_test"], m, s, cfg,
                        results_path(cfg, f"cert_{_tag(cfg)}_s{s}.jsonl"))


# ------------------------------------------------------------------------------------------------ full test split
def exp_fulltest(cfg, methods=FIXED_METHODS, batch=125):
    """Complete test split (7,180 PathMNIST images): clean accuracy, single-draw noisy accuracy and majority vote over
    cfg.full_votes noisy draws (a Monte-Carlo approximation of the smoothed prediction without abstention). One image
    pass per noise draw is shared by all methods."""
    out = results_path(cfg, f"fulltest_{_tag(cfg)}.npz")
    if os.path.exists(out):
        print("[fulltest] done:", out)
        return
    vlm, data = _setup(cfg)
    y = data["y_full"]
    C = len(data["classnames"])
    arrays = dict(labels=y)
    for s in cfg.sigmas:
        txt = build(vlm, data, cfg, s, methods)          # cached prompts (trained in `certify`)
        clean = {k: [] for k in txt}
        votes = {k: np.zeros((len(y), C), dtype=np.int16) for k in txt}
        i0 = 0
        with torch.no_grad():
            for x, _ in iterate_full(data, batch, vlm.device):
                f = vlm.encode_image(x)
                for k, t in txt.items():
                    clean[k].append((f @ t.t()).argmax(-1).cpu().numpy())
                for _ in range(cfg.full_votes):
                    f = vlm.encode_image(x + s * torch.randn_like(x))
                    for k, t in txt.items():
                        p = (f @ t.t()).argmax(-1).cpu().numpy()
                        votes[k][np.arange(i0, i0 + len(p)), p] += 1
                i0 += len(x)
        for k in txt:
            arrays[f"clean|{k}|{s}"] = np.concatenate(clean[k]).astype(np.int16)
            arrays[f"votes|{k}|{s}"] = votes[k]
        print(f"[fulltest] sigma={s}: " + ", ".join(
            f"{k}={100 * (votes[k].argmax(1) == y).mean():.1f}%" for k in txt))
    np.savez_compressed(out, **arrays)


# ------------------------------------------------------------------------------------------------ [3] attacks
def exp_attacks(cfg, smooth_methods=("zs_plip", "ps_fewshot", "coreprompt"), smooth_sigmas=(0.25, 0.5), bs=16):
    vlm, data = _setup(cfg)
    x_all, y_all = data["x_test"][:cfg.n_attack], data["y_test"][:cfg.n_attack]
    out_p = results_path(cfg, f"attacks_{_tag(cfg)}.json")
    res = load_json(out_p) if os.path.exists(out_p) else {}

    # (a) input-transformation defences on zero-shot PLIP: non-adaptive (transfer) vs BPDA/EOT adaptive PGD-Linf
    t0 = zeroshot_text_feats(vlm, data)
    undefended = A.base_logits_fn(vlm, t0, "none")
    for dname in ["none", "bit_depth", "jpeg", "resize_pad"]:
        defended = A.base_logits_fn(vlm, t0, dname)
        rep = 5 if dname == "resize_pad" else 1
        eot = cfg.eot if dname == "resize_pad" else 1
        for eps in [0.0] + list(cfg.linf_eps) + [16 / 255]:          # 16/255 = sanity check (should approach 0%)
            for mode in (["clean"] if eps == 0 else ["nonadaptive", "adaptive"]):
                key = f"transform|{dname}|{mode}|linf={eps:.5f}"
                if key in res:
                    continue
                accs = []
                for i in range(0, len(y_all), bs):
                    x = x_all[i:i + bs].to(vlm.device); y = y_all[i:i + bs].to(vlm.device)
                    if mode == "nonadaptive":
                        x = A.pgd(undefended, x, y, eps, cfg.pgd_steps, "linf")
                    elif mode == "adaptive":
                        x = A.pgd(defended, x, y, eps, cfg.pgd_steps, "linf", eot=eot)
                    accs.append(A.base_accuracy(defended, x, y, rep).cpu())
                res[key] = float(torch.cat(accs).mean())
                print(key, round(res[key], 3)); save_json(res, out_p)

    # (b) smoothed classifiers: non-adaptive PGD-L2 on the base model vs adaptive SmoothAdv (EOT over the noise)
    for s in smooth_sigmas:
        meth = build(vlm, data, cfg, s, list(smooth_methods))
        for name, t in meth.items():
            base = A.base_logits_fn(vlm, t, "none")
            for eps in [0.0] + list(cfg.l2_eps):
                for mode in (["clean"] if eps == 0 else ["nonadaptive", "adaptive"]):
                    key = f"smoothed|{name}|sigma={s}|{mode}|l2={eps}"
                    if key in res:
                        continue
                    correct = []
                    for i in range(0, len(y_all), bs):
                        x = x_all[i:i + bs].to(vlm.device); y = y_all[i:i + bs].to(vlm.device)
                        if mode == "nonadaptive":
                            x = A.pgd(base, x, y, eps, cfg.pgd_steps, "l2")
                        elif mode == "adaptive":
                            x = A.pgd(base, x, y, eps, cfg.pgd_steps, "l2", eot=cfg.eot, noise_sigma=s)
                        for j in range(len(y)):
                            pr = A.predict_smoothed(vlm, t, x[j], s, cfg.predict_n, cfg.alpha, cfg.cert_batch)
                            correct.append(int(pr == int(y[j])))
                    res[key] = float(np.mean(correct))
                    print(key, round(res[key], 3)); save_json(res, out_p)
    save_json(res, out_p)


# ------------------------------------------------------------------------------------------------ [4] ablation
ABLATION_GRID = {
    "lam_con": [0.0, 5.0, 20.0],
    "eta_ent": [0.0, 10.0],               # 10.0 = lambda: Jeong & Shin's original equal weighting
    "mu_rad": [0.0, 0.5, 2.0],
    "gamma_rad": [2.0, 8.0],              # 8.0 = MACER default
    "beta_rad": [0.5, 2.0],
    "m_train": [1, 2, 8],
    "shots": [4, 8],
}


def exp_ablation(cfg, sigma=0.5):
    """(i) loss-term ablation (CE / +con / +rad / full) and (ii) one-at-a-time sensitivity around the defaults."""
    vlm, data = _setup(cfg)
    base = build(vlm, data, cfg, sigma, ["ps_fewshot", "core_con", "core_rad", "coreprompt"])
    methods = {"abl_ce": base["ps_fewshot"], "abl_con": base["core_con"], "abl_rad": base["core_rad"],
               "abl_full": base["coreprompt"]}
    if cfg.grid == "small":
        grid = {k: v[:1] for k, v in ABLATION_GRID.items()}
    elif cfg.grid == "medium":      # one informative value per hyper-parameter (incl. the literature defaults)
        grid = dict(lam_con=[20.0], eta_ent=[10.0], mu_rad=[2.0], gamma_rad=[8.0], beta_rad=[2.0], m_train=[2],
                    shots=[4])
    else:
        grid = ABLATION_GRID
    for key, values in grid.items():
        for v in values:
            c2 = dataclasses.replace(cfg, **{key: v})
            d2 = data
            if key == "shots":   # stratified subset of the 16-shot set
                keep = torch.cat([(data["y_shot"] == c).nonzero().squeeze(1)[:v] for c in range(len(data["classnames"]))])
                d2 = {**data, "x_shot": data["x_shot"][keep], "y_shot": data["y_shot"][keep]}
            ctx, _ = train_prompt(vlm, d2, c2, "coreprompt", sigma, init="noisy_text", tag=f"_{key}{v}", verbose=False)
            methods[f"sens|{key}|{v}"] = text_feats_for(vlm, data, ctx)
    for n_ctx in ({"small": [4], "medium": [8]}.get(cfg.grid, [4, 8, 16])):      # context length, CoOp-style random initialisation
        ctx, _ = train_prompt(vlm, data, cfg, "coreprompt", sigma, init="random", n_ctx=n_ctx, tag=f"_rand{n_ctx}",
                              verbose=False)
        methods[f"sens|n_ctx_random|{n_ctx}"] = text_feats_for(vlm, data, ctx)
    certify_methods(vlm, data["x_test"], data["y_test"], methods, sigma, cfg,
                    results_path(cfg, f"ablation_{_tag(cfg)}_s{sigma}.jsonl"))


# ------------------------------------------------------------------------------------------------ [5] generalisation
def exp_gen_clip(cfg):
    exp_certify(cfg, GEN_METHODS)
    exp_fulltest(cfg)


def exp_gen_blood(cfg):
    exp_certify(cfg, GEN_METHODS)
    exp_fulltest(cfg)


# ------------------------------------------------------------------------------------------------ efficiency
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def exp_efficiency(cfg):
    from .smoothing import noisy_feature_counts
    vlm, data = _setup(cfg)
    total = sum(p.numel() for p in vlm.clip.parameters())
    s = cfg.sigmas[0]
    logs = {}
    meth = build(vlm, data, cfg, s, ALL_METHODS, logs=logs)
    rows = [dict(method=k, trainable_params=lg["trainable_params"], frozen_params=total,
                 train_time_s=lg.get("train_time_s"), peak_train_mem_mb=lg.get("peak_mem_mb")) for k, lg in logs.items()]
    x = data["x_test"][0].to(vlm.device)
    fixed = [v for v in meth.values() if not callable(v)]
    noisy_feature_counts(vlm, x, fixed[:1], s, 100, cfg.cert_batch)        # warm-up
    _sync(); t0 = time.time(); noisy_feature_counts(vlm, x, fixed[:1], s, cfg.n, cfg.cert_batch); _sync()
    single = time.time() - t0
    _sync(); t0 = time.time(); noisy_feature_counts(vlm, x, fixed, s, cfg.n, cfg.cert_batch); _sync()
    shared = time.time() - t0
    _sync(); t0 = time.time(); meth["ps_combined"](x); _sync()
    tta = time.time() - t0
    res = dict(rows=rows, cert_s_per_image_one_method=single, cert_s_per_image_all_fixed_shared=shared,
               n_fixed_methods=len(fixed), tta_s_per_image=tta, n=cfg.n,
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
    save_json(res, results_path(cfg, "efficiency.json"))
    print(json.dumps(res, indent=1))


# ------------------------------------------------------------------------------------------------ report
def exp_report(cfg):
    from .report import make_report
    make_report(cfg.out_dir)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=list(STAGES.keys()))
    ap.add_argument("--preset", default="colab")
    ap.add_argument("--out_dir", default="outputs")
    ap.add_argument("--data_dir", default=None)
    ap.add_argument("--set", nargs="*", default=[], help="config overrides key=value")
    a = ap.parse_args()
    ov = {}
    for kv in a.set:
        k, v = kv.split("=", 1)
        try:
            ov[k] = ast.literal_eval(v)
        except (ValueError, SyntaxError):
            ov[k] = v
    cfg = make_config(a.preset, a.stage, out_dir=a.out_dir, data_dir=a.data_dir, **ov)
    t0 = time.time()
    globals()[f"exp_{a.stage}"](cfg)
    minutes = (time.time() - t0) / 60
    tp = results_path(cfg, "timings.json")
    tim = load_json(tp) if os.path.exists(tp) else {}
    if a.stage != "report":
        tim[a.stage] = round(minutes + tim.get(a.stage, 0.0), 2)    # accumulates across resumed sessions
        save_json(tim, tp)
    print(f"[stage {a.stage}] finished in {minutes:.1f} min")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tests/__init__.py


In [ ]:
%%writefile tests/test_math.py
"""Unit tests for the certification and loss mathematics (CPU only; run with `python -m pytest tests -q`)."""
import numpy as np
import torch
from scipy.stats import norm

from coreprompt.losses import consistency, radius_hinge, ce_noise, entropy_of_mean
from coreprompt.smoothing import (ABSTAIN, certify_from_counts, clopper_pearson_lower, max_certifiable_radius,
                                  certified_accuracy, acr)
from coreprompt import defenses
from coreprompt.attacks import _project_l2


def test_clopper_pearson_against_known_value():
    # all n successes: lower bound = alpha^(1/n)
    assert abs(clopper_pearson_lower(100, 100, 0.001) - 0.001 ** (1 / 100)) < 1e-9
    assert clopper_pearson_lower(0, 100, 0.001) == 0.0
    # lower bound is below the empirical proportion and increases with n at fixed proportion
    assert clopper_pearson_lower(80, 100, 0.001) < 0.8
    assert clopper_pearson_lower(800, 1000, 0.001) > clopper_pearson_lower(80, 100, 0.001)


def test_max_radius():
    r = max_certifiable_radius(2000, 0.001, 0.25)
    assert abs(r - 0.25 * norm.ppf(0.001 ** (1 / 2000))) < 1e-12
    assert 0.6 < r < 0.75          # ~2.7 sigma


def test_certify_abstain_and_radius():
    c0 = np.array([0, 60, 4]); c1 = np.array([10, 1990, 0])
    pred, rad, pl = certify_from_counts(c0, c1, 2000, 0.001, 0.5)
    assert pred == 1 and rad > 0 and abs(rad - 0.5 * norm.ppf(pl)) < 1e-12
    pred, rad, _ = certify_from_counts(c0, np.array([1000, 1000, 0]), 2000, 0.001, 0.5)
    assert pred == ABSTAIN and rad == 0.0


def test_certified_accuracy_and_acr():
    import pandas as pd
    df = pd.DataFrame(dict(correct=[1, 1, 0, 1], radius=[0.1, 0.6, 0.9, 0.0]))
    ca = certified_accuracy(df, [0.0, 0.5, 1.0])
    assert ca[0.0] == 0.75 and ca[0.5] == 0.25 and ca[1.0] == 0.0
    assert abs(acr(df) - 0.7 / 4) < 1e-12


def test_consistency_zero_when_identical():
    z = torch.randn(1, 5, 9).repeat(4, 1, 1)
    loss, kl, ent = consistency(z, 10.0, 0.0)
    assert kl.abs() < 1e-6 and loss.abs() < 1e-5


def test_consistency_positive_when_different():
    z = torch.randn(4, 5, 9)
    _, kl, _ = consistency(z, 1.0, 0.0)
    assert kl > 0


def test_radius_hinge_gradient_increases_margin():
    torch.manual_seed(0)
    y = torch.tensor([0, 1, 2])
    z = torch.randn(4, 3, 5) * 0.1
    z[:, torch.arange(3), y] += 1.0                         # correct but low margin
    z.requires_grad_(True)
    loss, r0 = radius_hinge(z, y, sigma=0.5, gamma=8.0, beta=1.0)
    loss.backward()
    with torch.no_grad():
        z2 = z - 1.0 * z.grad
    _, r1 = radius_hinge(z2, y, sigma=0.5, gamma=8.0, beta=1.0)
    assert r1 > r0                                           # descent step enlarges the soft certified radius


def test_radius_hinge_ignores_misclassified():
    y = torch.tensor([0])
    z = torch.zeros(2, 1, 3); z[:, 0, 1] = 5.0                # predicts class 1, label 0
    loss, _ = radius_hinge(z, y, 0.5, 2.0, 1.0)
    assert loss.item() == 0.0


def test_ce_and_entropy_shapes():
    z = torch.randn(3, 4, 6); y = torch.randint(0, 6, (4,))
    assert ce_noise(z, y).ndim == 0 and entropy_of_mean(z).ndim == 0


def test_bpda_identity_gradient():
    x = torch.rand(2, 3, 8, 8, requires_grad=True)
    out = defenses.apply("bit_depth", x)
    out.sum().backward()
    assert torch.allclose(x.grad, torch.ones_like(x))
    assert torch.allclose(out, defenses.bit_depth(x.detach()))


def test_resize_pad_shape_and_jpeg_range():
    x = torch.rand(2, 3, 32, 32)
    assert defenses.resize_pad(x).shape == x.shape
    j = defenses.jpeg(x)
    assert j.shape == x.shape and j.min() >= 0 and j.max() <= 1


def test_l2_projection():
    d = torch.randn(4, 3, 8, 8) * 10
    p = _project_l2(d, 0.5)
    assert torch.all(p.flatten(1).norm(dim=1) <= 0.5 + 1e-5)

In [ ]:
%%writefile tests/fake_results.py
"""Writes synthetic result files in exactly the formats produced by coreprompt.run (for testing the report offline)."""
import json, os, sys
import numpy as np
from scipy.stats import norm

def write(out, n=60, N_full=300, seed=0):
    rng = np.random.default_rng(seed)
    res = os.path.join(out, "results"); os.makedirs(res, exist_ok=True)
    strength = dict(zs_plip=0.2, coop=0.5, ps_zeroshot=0.45, ps_fewshot=0.9, ps_combined=1.2, coreprompt=1.3,
                    coreprompt_tta=1.5, core_con=1.0, core_rad=1.1)
    def cert(path, methods, sigmas, C=9):
        with open(path, "a") as f:
            for i in range(n):
                y = int(rng.integers(C)); h = rng.normal()
                for m in methods:
                    for s in sigmas:
                        pass
                for m in methods:
                    st = strength.get(m.split("|")[0], 1.0) if not m.startswith("sens") else 1.2 + 0.1 * rng.normal()
                    for s in sigmas:
                        pass
                f.write("")
        # per sigma files written by caller
    def cert_file(path, methods, s, C=9):
        with open(path, "w") as f:
            for i in range(n):
                y = int(rng.integers(C)); h = rng.normal()
                for m in methods:
                    st = strength.get(m, 1.2 + 0.2 * rng.normal())
                    z = st + h + 0.3 * rng.normal() - 0.8 * s
                    pA = min(norm.cdf(z + 0.5), 0.9995)
                    if pA < 0.5:
                        pred, rad = -1, 0.0
                    else:
                        pred = y if rng.random() < 0.85 else (y + 1) % C
                        rad = s * norm.ppf(pA)
                    f.write(json.dumps(dict(idx=i, method=m, sigma=s, label=y, predict=pred, radius=rad, pA_lower=pA,
                                            correct=int(pred == y), time_s=1.0)) + "\n")
    allm = ["zs_plip", "coop", "ps_zeroshot", "ps_fewshot", "ps_combined", "coreprompt", "coreprompt_tta"]
    gen = ["zs_plip", "coop", "ps_fewshot", "ps_combined", "coreprompt", "coreprompt_tta"]
    for s in [0.1, 0.25, 0.5, 1.0]:
        cert_file(f"{res}/cert_pathmnist_plip_s{s}.jsonl", allm, s)
    for ds, model, C in [("pathmnist", "clip", 9), ("bloodmnist", "plip", 8)]:
        for s in [0.25, 0.5, 1.0]:
            cert_file(f"{res}/cert_{ds}_{model}_s{s}.jsonl", gen, s, C)
    sens = [f"sens|{k}|{v}" for k, vs in dict(lam_con=[0, 5, 20], eta_ent=[0, 10], mu_rad=[0, 0.5, 2], gamma_rad=[2, 8],
            beta_rad=[0.5, 2], m_train=[1, 2, 8], shots=[4, 8], n_ctx_random=[4, 8, 16]).items() for v in vs]
    strength.update(abl_ce=0.9, abl_con=1.0, abl_rad=1.1, abl_full=1.3)
    cert_file(f"{res}/ablation_pathmnist_plip_s0.5.jsonl", ["abl_ce", "abl_con", "abl_rad", "abl_full"] + sens, 0.5)
    for ds, model, C, sig in [("pathmnist", "plip", 9, [0.1, 0.25, 0.5, 1.0]), ("pathmnist", "clip", 9, [0.25, 0.5, 1.0]),
                              ("bloodmnist", "plip", 8, [0.25, 0.5, 1.0])]:
        y = rng.integers(0, C, N_full); arr = dict(labels=y)
        for m in ["zs_plip", "coop", "ps_fewshot", "coreprompt"]:
            for s in sig:
                p = 0.9 - 0.3 * s * (1.5 - strength[m] / 1.5)
                arr[f"clean|{m}|{s}"] = np.where(rng.random(N_full) < 0.9, y, (y + 1) % C).astype(np.int16)
                v = np.zeros((N_full, C), np.int16)
                for _ in range(16):
                    pr = np.where(rng.random(N_full) < p, y, rng.integers(0, C, N_full)); v[np.arange(N_full), pr] += 1
                arr[f"votes|{m}|{s}"] = v
        np.savez_compressed(f"{res}/fulltest_{ds}_{model}.npz", **arr)
    att = {}
    for d in ["none", "bit_depth", "jpeg", "resize_pad"]:
        att[f"transform|{d}|clean|linf=0.00000"] = 0.8
        for e in [1, 2, 4, 16]:
            att[f"transform|{d}|nonadaptive|linf={e / 255:.5f}"] = max(0.8 - 0.1 * e, 0.05) + (0.2 if d != "none" else 0)
            att[f"transform|{d}|adaptive|linf={e / 255:.5f}"] = max(0.8 - 0.15 * e, 0.0)
    for s in [0.25, 0.5]:
        for m in ["zs_plip", "ps_fewshot", "coreprompt"]:
            att[f"smoothed|{m}|sigma={s}|clean|l2=0.0"] = 0.7
            for e in [0.25, 0.5, 1.0]:
                att[f"smoothed|{m}|sigma={s}|nonadaptive|l2={e}"] = 0.7 - 0.1 * e
                att[f"smoothed|{m}|sigma={s}|adaptive|l2={e}"] = 0.7 - 0.3 * e
    json.dump(att, open(f"{res}/attacks_pathmnist_plip.json", "w"))
    lg = lambda m, s: dict(method=m, sigma=s, trainable_params=4096, epoch_loss=list(np.exp(-np.arange(50) / 10) + 0.5),
                           soft_radius=[None] * 50 if m != "coreprompt" else list(0.1 + 0.01 * np.arange(50)),
                           train_time_s=40.0, peak_mem_mb=900.0)
    json.dump({str(s): {m: lg(m, s) for m in ["coop", "ps_fewshot", "coreprompt"]} for s in [0.1, 0.25, 0.5, 1.0]},
              open(f"{res}/trainlogs_pathmnist_plip.json", "w"))
    json.dump(dict(rows=[dict(method=m, trainable_params=4096, frozen_params=151e6, train_time_s=40.0, peak_train_mem_mb=900)
                         for m in ["coop", "ps_fewshot", "coreprompt"]], cert_s_per_image_one_method=1.1,
                   cert_s_per_image_all_fixed_shared=1.3, n_fixed_methods=4, tta_s_per_image=0.2, n=2000, gpu="Tesla T4"),
              open(f"{res}/efficiency.json", "w"))
    json.dump(dict(min_cosine_learnable_vs_reference=0.99999, zeroshot_clean_acc=0.62, n_images=500, img_size=224,
                   max_radius_at_sigma={"0.1": 0.27, "0.25": 0.67}), open(f"{res}/sanity.json", "w"))
    for ds, C in [("pathmnist", 9), ("bloodmnist", 8)]:
        json.dump(dict(dataset=ds, classes=[f"class {c}" for c in range(C)], img_size=224, shots=16, n_shot_images=16 * C,
                       test_split_total=7180, test_class_counts=[100] * C, certified_subset_class_counts=[10] * C),
                  open(f"{res}/datastats_{ds}.json", "w"))
    json.dump(dict(sanity=2.0, certify=60.0, attacks=30.0), open(f"{res}/timings.json", "w"))

if __name__ == "__main__":
    write(sys.argv[1])

In [ ]:
%%writefile tests/test_report.py
"""Report generation on synthetic results in the exact on-disk formats (no GPU, no torch needed)."""
import json
import os

from coreprompt.report import make_report
from tests.fake_results import write


def test_report_end_to_end(tmp_path):
    write(str(tmp_path))
    idx = make_report(str(tmp_path))
    sections = {i["section"] for i in idx}
    for must in ["4.3 Baseline reproduction", "4.5 Ablation", "4.5 Sensitivity", "4.5 Generalisation",
                 "4.5 Robustness — adaptive vs non-adaptive", "4.5 Efficiency"]:
        assert must in sections, must
    for it in idx:
        for k in ("figure", "table"):
            if it.get(k):
                assert os.path.exists(os.path.join(tmp_path, "report", it[k])), it[k]
    assert json.load(open(tmp_path / "report" / "index.json"))

In [ ]:
%%writefile tests/test_data_remote.py
"""Remote .npz reading over HTTP range requests against a local range-capable server (stored and compressed)."""
import http.server
import os
import re
import threading
from types import SimpleNamespace

import numpy as np

from coreprompt import data as D


class _H(http.server.SimpleHTTPRequestHandler):
    def do_GET(self):
        path = self.translate_path(self.path.split("?")[0])
        size = os.path.getsize(path)
        m = re.match(r"bytes=(\d+)-(\d*)", self.headers.get("Range", ""))
        if not m:
            return super().do_GET()
        a, b = int(m.group(1)), min(int(m.group(2) or size - 1), size - 1)
        self.send_response(206)
        self.send_header("Content-Range", f"bytes {a}-{b}/{size}")
        self.send_header("Content-Length", str(b - a + 1))
        self.end_headers()
        with open(path, "rb") as f:
            f.seek(a)
            self.wfile.write(f.read(b - a + 1))

    def log_message(self, *a):
        pass


def test_remote_sample(tmp_path):
    rng = np.random.default_rng(0)
    arrays = {}
    for sp, n in [("train", 40), ("val", 200), ("test", 300)]:
        arrays[f"{sp}_images"] = rng.integers(0, 255, (n, 16, 16, 3), dtype=np.uint8)
        arrays[f"{sp}_labels"] = rng.integers(0, 9, (n, 1))
    np.savez(tmp_path / "stored.npz", **arrays)
    np.savez_compressed(tmp_path / "deflated.npz", **arrays)
    handler = lambda *a, **k: _H(*a, directory=str(tmp_path), **k)
    srv = http.server.ThreadingHTTPServer(("127.0.0.1", 0), handler)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    try:
        for name in ["stored", "deflated"]:
            url = f"http://127.0.0.1:{srv.server_port}/{name}.npz?download=1"
            d = D._sample("pathmnist", 16, 4, 0, url=url)
            assert np.array_equal(d["x_test"], arrays["test_images"])
            assert np.array_equal(d["x_shot"], arrays["val_images"][d["shot_idx"]])
            assert np.array_equal(d["y_shot"], arrays["val_labels"].reshape(-1)[d["shot_idx"]])
            cfg = SimpleNamespace(data_dir=str(tmp_path / f"cache_{name}"), out_dir=str(tmp_path), dataset="pathmnist",
                                  img_size=16, seed=0, shots=4, n_test=50, n_full=None, data_url=url)
            out = D.load_data(cfg)
            assert out["x_test"].shape == (50, 3, 16, 16) and len(out["y_full"]) == 300
            assert np.array_equal(out["y_test"].numpy(), arrays["test_labels"].reshape(-1)[out["stats"]["test_idx"]])
            # reduced budgets read the small head cache: must give exactly the same certified images
            cfg.n_full, cfg.n_test = 40, 30
            head = D.load_data(cfg)
            assert np.array_equal(head["x_test"].numpy(), out["x_test"].numpy()[:30])
            assert head["stats"]["test_idx"] == out["stats"]["test_idx"][:30] and len(head["y_full"]) == 40
            ids = np.array(head["stats"]["test_idx"])
            assert np.array_equal(head["y_test"].numpy(), arrays["test_labels"].reshape(-1)[ids])
    finally:
        srv.shutdown()

In [ ]:
import coreprompt; print('coreprompt version', coreprompt.__version__, '(should be 1.2.0)')

## Helper: run one stage in a fresh process and stop the notebook if it fails

In [ ]:
import subprocess, sys, time
def run_stage(stage, preset='quick', out_dir=None):
    out_dir = out_dir or OUT_DIR
    cmd = [sys.executable, '-m', 'coreprompt.run', stage, '--preset', preset, '--out_dir', out_dir, '--data_dir', DATA_DIR]
    print('>>>', ' '.join(cmd[2:]), flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'stage {stage} ({preset}) failed - see the traceback above. '
                           'Fix or report it, then Run all again (finished work is kept).')
    print(f'<<< {stage} ok ({(time.time() - t0) / 60:.1f} min)')

STAGES = ['sanity', 'datastats', 'certify', 'fulltest', 'attacks', 'ablation', 'gen_clip', 'gen_blood', 'efficiency']

## Step 0 — unit tests (30 s)

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('unit tests failed - see output above')

## Step 1 — sanity checks and data statistics (Sections 3.1, 4.3)

In [ ]:
run_stage('sanity')
run_stage('datastats')

## Step 2 — [1] PromptSmooth reproduction + [2] CoRe-Prompt certification (Sections 4.3, 4.4)
Trains CoOp, PromptSmooth few-shot and CoRe-Prompt prompts at each σ (cached in Drive) and certifies all seven methods on the same 100 images and noise. Progress every 50 images; resumes per image.

In [ ]:
run_stage('certify')

## Step 3 — [2] all fixed-prompt methods on 2,000 test images

In [ ]:
run_stage('fulltest')

## Step 4 — [3] adaptive vs non-adaptive attacks (Section 4.5)

In [ ]:
run_stage('attacks')

## Step 5 — [4] ablation and sensitivity (Section 4.5)

In [ ]:
run_stage('ablation')

## Step 6 — [5] generalisation: CLIP ViT-B/32 (cross-model) and BloodMNIST (cross-dataset)

In [ ]:
run_stage('gen_clip')
run_stage('gen_blood')

## Step 7 — computational efficiency

In [ ]:
run_stage('efficiency')

## Step 8 — results: tables, figures and statistical tests
Everything below is also saved to `OUT_DIR/report/` (PNG + PDF figures, CSV + LaTeX tables).

In [ ]:
run_stage('report')
from coreprompt.report import show
show(OUT_DIR)

## Step 9 — download
The zip contains `results/` (raw) and `report/` (tables + figures). Send it back to write Section 4.

In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M')
tmp = f'/content/coreprompt_results_{stamp}'
os.makedirs(tmp, exist_ok=True)
for sub in ['results', 'report']:
    src = os.path.join(OUT_DIR, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(tmp, sub), dirs_exist_ok=True)
zip_path = shutil.make_archive(tmp, 'zip', tmp)
print(zip_path, f'{os.path.getsize(zip_path) / 2**20:.1f} MB')
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('download it from the Files panel:', zip_path, e)